# From Waveform to Words: Training an HMM/GMM Speech Recognizer

*Tutorial version 2.0, 2026-09-05.*

*With thanks to Esther Klabbers.*

This notebook builds a working speech recognizer from raw audio. You will:

- inspect a speech waveform and turn it into the features an acoustic
  model trains on (spectrogram → mel filterbank → MFCCs),
- build a pronunciation dictionary and a training corpus from a public
  speech database the notebook downloads,
- train a context-dependent HMM/GMM acoustic model with `pstrain`
  (a modern, from-scratch reimplementation of the classic CMU SphinxTrain
  pipeline),
- use that model to force-align known transcripts to audio and to decode
  unseen audio with PocketSphinx,
- package the trained model into a distributable form and re-test it.

**Prerequisites.** Comfort with NumPy array manipulation and a first course
covering probability (expectation, Gaussian densities) and, ideally, a brief
prior exposure to Markov chains.

No prior speech-processing background is assumed. Every signal-processing
and modeling concept is introduced from first principles, with a small
from-scratch NumPy demo before we hand the real computation to `pstrain`.

**Requirements.** A Python environment with internet access. The notebook
installs its dependencies from PyPI and downloads its own copy of the
CMU\_ARCTIC speech database; it depends on no external repository, cached
file, or bundled fixture.

CMU\_ARCTIC was built at Carnegie Mellon University's Language Technologies
Institute as phonetically balanced, single-speaker US English recordings for
speech synthesis and recognition research, and is freely redistributable;
see <http://festvox.org/dbs/index.html>.

## How to read this notebook

The notebook follows the order a real acoustic-model training project
would, one section at a time:

- **Environment setup** installs the packages and sets the handful of knobs
  everything downstream reads.
- **The speech data** downloads the corpus, builds a pronunciation
  dictionary, and reads the transcripts.
- **Speech Analysis** walks the waveform to the spectrum to the mel
  filterbank to the MFCCs an acoustic model actually sees.
- **Project setup** turns raw prompts into a training-ready transcript and
  lays out a `pstrain` project.
- **Features and configuration** pins the front-end contract that training
  and decoding must agree on.
- **The model** introduces the HMM topology, the Gaussian mixtures, and the
  model families you can ask for.
- **Training** runs the real pipeline, then takes each step apart with a
  small from-scratch demonstration.
- **Alignment and decoding** puts the trained model to work, first on known
  transcripts and then on unknown ones.
- **Model capacity and configuration** lays out the levers that trade data,
  compute, and accuracy against each other.
- **Packaging and deployment** ships the model and smoke-tests what was
  shipped.

Recurring boxes appear along the way:

> **Checkpoint** — a short conceptual question. Try answering it yourself
> before expanding the answer.

> **Try it yourself** — a small modification to make and re-run, to build
> intuition by seeing what changes. These are optional but worthwhile.

Most cells run in a few seconds. The cells that do the corpus download and
the actual model training are marked **HEAVY**:

- the corpus download takes anywhere from a few seconds to a couple of
  minutes, depending on your connection;
- the default training configuration (150 utterances, target `cd-1g`) takes
  roughly 8–15 seconds on a modern laptop.

Re-running the notebook top to bottom after the first pass is fast:
`pstrain` skips any stage whose inputs haven't changed since it last ran
successfully, and the corpus download is cached on disk after the first
run.

# Environment setup

Everything this notebook needs is installable from PyPI:

| package | role |
|---|---|
| `pstrain` | the HMM/GMM training toolkit — feature extraction, Baum–Welch training, decision-tree state tying, forced alignment, model packaging |
| `pocketsphinx` | the CPU speech decoder `pstrain` drives for testing and forced alignment |
| `cmudict` | the CMU Pronouncing Dictionary, used to build our word → phoneme dictionary |
| `librosa` | loads audio and computes spectrograms/MFCCs for the illustrative signal-processing demos |
| `jiwer` | an independent, widely-used implementation of word error rate (WER) and its character-level counterpart, used in Alignment and decoding to cross-check `pstrain`'s own reported error rate |
| `numpy`, `matplotlib` | array math and plotting |

This notebook installs the latest pstrain from PyPI by default. It was
checked against 0.4.0.

In [ ]:
# Tested on pstrain 0.4.0. The notebook tracks the released package rather
# than pinning a version, so a newer release is picked up automatically.
import os
import subprocess
import sys

# CI runs this notebook offline against the pstrain checkout it is testing,
# where every package below is already installed, so pip is skipped there.
if os.environ.get("PSTRAIN_TUTORIAL_OFFLINE") == "1":
    try:
        import cmudict
        import jiwer
        import librosa
        import matplotlib
        import pocketsphinx
        import pstrain
    except ModuleNotFoundError as error:
        raise RuntimeError(
            f"PSTRAIN_TUTORIAL_OFFLINE=1 needs {error.name}, which is not "
            "installed. Install it, or unset the variable to let this cell "
            "install the dependencies itself."
        ) from error
    print("Offline mode: using the preinstalled packages")
else:
    # pip through subprocess rather than the %pip magic, whose own
    # "you may need to restart the kernel" note fires on every run.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pstrain",
            "pocketsphinx>=5.0",
            "cmudict",
            "librosa",
            "matplotlib",
            "numpy",
            "jiwer",
        ],
        check=True,
    )
    print("Restart the kernel if you update the packages.")

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import librosa
import librosa.display
import pstrain
import pocketsphinx
import jiwer
from importlib.metadata import version as pkg_version

print(f"numpy        {np.__version__}")
print(f"matplotlib   {matplotlib.__version__}")
print(f"librosa      {librosa.__version__}")
print(f"jiwer        {pkg_version('jiwer')}")
print(f"pstrain      {pstrain.__version__}  ({Path(pstrain.__file__).resolve()})")
print(f"pocketsphinx {pkg_version('pocketsphinx')}")

## Configuration

Everything downstream is controlled by the handful of settings below. The
defaults give a fast, complete run (roughly 8–15 seconds of actual training
on a modern many-core laptop); raising `N_UTTS` and the target density gives
a more accurate, and slower, model.

`JOBS` sets how many worker processes `pstrain` may use. It does not speed
every stage up equally, because not every stage is parallel:

- feature extraction is one independent task per utterance, so it spreads
  across `JOBS` workers;
- the decision trees that tie context-dependent states are one task per
  tree, and likewise parallelize;
- the Baum-Welch passes, the iterative re-estimation that actually fits the
  model, run serially while `training.multipron_training` is on, meaning every listed pronunciation of a word stays live in the
  training graph, which is `pstrain`'s current behavior and this notebook's
  setting;
- decoding is one utterance per worker, so it parallelizes too.

`JOBS = 1` remains a fine choice on a small machine, and offline/CI runs use
it. From a terminal, the same run parallelizes with `pstrain build ... -j N`.

In [ ]:
import os

WORK = Path.cwd() / "arctic_tutorial_work"
WORK.mkdir(parents=True, exist_ok=True)

# CI sets this to run the notebook with no network at all: the package
# install above is skipped, and the ten-utterance corpus bundled with a
# pstrain checkout stands in for the CMU_ARCTIC download below.
OFFLINE = os.environ.get("PSTRAIN_TUTORIAL_OFFLINE") == "1"

VOICE = "slt"       # CMU_ARCTIC US-English female voice
# How many transcribed utterances to train on. Set N_UTTS = None to use
# the WHOLE corpus -- pstrain is fast enough that this is cheap. Measured
# end to end on a laptop: 150 utterances ~35s, the whole corpus ~75s, of
# which training is ~47s. Feature extraction costs the same either way,
# because it runs over every utterance in the linked audio regardless.
N_UTTS = 150
TARGET = "cd-1g"     # training target: context-dependent, 1 Gaussian/state
# Parallel training and decoding workers; offline/CI runs keep 1.
JOBS = 1 if OFFLINE else min(4, os.cpu_count() or 1)
FORCE = False        # set True to force a clean rebuild of every stage

# Training survives an utterance it can't force-align: it reports the
# utterance and omits it from that pass (see the Training section).
# arctic_a0135 is a known short utterance in this corpus that this happens
# to. List an id here only to keep it out of the plots and counts entirely.
EXCLUDE_UTTERANCES = []

np.random.seed(42)
print("Work directory:", WORK)
print("Utterances:", "all" if N_UTTS is None else N_UTTS,
      "| target:", TARGET, "| jobs:", JOBS)
if OFFLINE:
    print("Offline mode: the bundled mini corpus stands in for CMU_ARCTIC")

# The speech data

We need two things before we can train anything: audio with matching
transcripts (the corpus), and a dictionary mapping the words in those
transcripts to sequences of phonemes (the pronunciations).

Both are acquired here, from scratch, so the rest of the notebook has real
data to work with.

> **Note.** If the download fails, the usual cause is a firewall or a
> campus/VPN proxy blocking outbound HTTP to festvox.org. Try another
> network (a phone hotspot works) or re-run the cell: transient failures are
> retried, and a completed archive is cached in `WORK` so you download
> once.

In [ ]:
import hashlib
import re
import socket
import tarfile
import time
import urllib.error
import urllib.parse
import urllib.request
import wave

ARCTIC_URL = "http://festvox.org/cmu_arctic/packed/cmu_us_slt_arctic.tar.bz2"
# The digest pstrain's own benchmark harness pins for this archive.
ARCTIC_SHA256 = "7c173297916acf3cc7fcab2713be4c60b27312316765a90934651d367226b4ea"
ARCTIC_ARCHIVE = WORK / "cmu_us_slt_arctic.tar.bz2"
CORPUS_ROOT = WORK / "corpus"
ARCTIC = CORPUS_ROOT / "cmu_us_slt_arctic"
WAV_DIR = ARCTIC / "wav"
IS_FULL = True       # False when OFFLINE substitutes the mini corpus


def resolve_download_host(url=ARCTIC_URL):
    '''Resolve the download host before connecting.

    A name that will not resolve and a host that will not answer are
    different failures with different fixes, so we separate them here and
    say which one happened.
    '''
    parts = urllib.parse.urlsplit(url)
    host = parts.hostname
    port = parts.port or (443 if parts.scheme == "https" else 80)
    try:
        socket.getaddrinfo(host, port)
    except socket.gaierror as exc:
        raise RuntimeError(
            f"DNS lookup for {host} failed ({exc}). That is name resolution "
            "failing, not the download: check your DNS, VPN, or proxy "
            "settings, then re-run this cell."
        ) from exc
    return host


def describe_download_failure(error, url, dest):
    '''Name which of three different failures the last attempt actually hit.

    The retry loop below sees all three through one except clause, and they
    need different fixes, so we tell them apart here rather than guessing.
    '''
    host = urllib.parse.urlsplit(url).hostname
    if isinstance(error, urllib.error.HTTPError):
        return (
            f"{host} answered with HTTP {error.code} ({error.reason}): the host "
            "is reachable but did not serve the archive. Check the URL, or try "
            "again later if the mirror is having trouble."
        )
    if isinstance(error, (urllib.error.URLError, TimeoutError, ConnectionError)):
        return (
            f"{host} resolved but the connection did not complete ({error}): a "
            "firewall or proxy is the usual cause. Try another network, then "
            "re-run this cell."
        )
    return (
        f"Writing {dest} failed ({error}): that is a local disk or permissions "
        f"problem, not a network one. Check space and permissions on "
        f"{dest.parent}, then re-run this cell."
    )


def download_arctic(url=ARCTIC_URL, dest=ARCTIC_ARCHIVE, attempts=3, timeout=60):
    '''Download the CMU_ARCTIC SLT archive, retrying transient failures.

    A partially-downloaded file is removed before each retry so we never
    resume into a corrupt archive. A fully-downloaded file is left in
    place and reused on the next run.
    '''
    if dest.exists():
        print(f"Already downloaded: {dest}")
        return dest
    host = resolve_download_host(url)
    print(f"{host} resolves; starting download")
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            print(f"Downloading {url} (attempt {attempt}/{attempts}) ...")
            urllib.request.urlretrieve(url, dest)
            return dest
        except (OSError, urllib.error.URLError) as exc:
            last_error = exc
            if dest.exists():
                dest.unlink()
            time.sleep(2 * attempt)
    raise RuntimeError(
        f"Could not download the CMU_ARCTIC SLT corpus from {url} after "
        f"{attempts} attempts. "
        + describe_download_failure(last_error, url, dest)
    ) from last_error


def find_mini_corpus():
    '''Locate the ten-utterance corpus bundled with a pstrain checkout.'''
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "tests/fixtures/mini_arctic"
        if (candidate / "wav").is_dir():
            return candidate
    raise RuntimeError(
        "PSTRAIN_TUTORIAL_OFFLINE=1 needs the mini corpus at "
        "tests/fixtures/mini_arctic, which ships only in a pstrain "
        "checkout. Unset the variable to download CMU_ARCTIC instead."
    )


if OFFLINE:
    # No network: train on the ten utterances that ship with the checkout.
    # Every later cell runs unchanged, just on a much smaller corpus.
    ARCTIC = find_mini_corpus()
    WAV_DIR = ARCTIC / "wav"
    IS_FULL = False
    print("Offline mode: using the bundled mini corpus at", ARCTIC)
else:
    download_arctic()
    digest = hashlib.sha256(ARCTIC_ARCHIVE.read_bytes()).hexdigest()
    if digest == ARCTIC_SHA256:
        print("SHA-256 matches the pin:", digest)
    else:
        # A warning, not a failure, so a mirror difference cannot stop a
        # class mid-session. That is the only reason: a mismatch is
        # unexplained until someone checks it.
        print("WARNING: archive SHA-256 does not match the pinned digest.")
        print("  expected:", ARCTIC_SHA256)
        print("  got:     ", digest)
        print("  A different mirror, a truncated download, and an altered "
              "archive all look like this. Continuing anyway.")

    if not WAV_DIR.exists():
        CORPUS_ROOT.mkdir(parents=True, exist_ok=True)
        with tarfile.open(ARCTIC_ARCHIVE, "r:bz2") as tf:
            tf.extractall(CORPUS_ROOT, filter="data")

n_wavs = len(list(WAV_DIR.glob("*.wav")))
if IS_FULL:
    print(f"Extracted {n_wavs} WAV files to {WAV_DIR}")
    assert n_wavs >= 1000, "Expected roughly 1,132 CMU_ARCTIC SLT utterances."
else:
    print(f"{n_wavs} WAV files in {WAV_DIR}")

A note on that SHA-256 check: the cell above compares the archive's digest
against the value `pstrain`'s own benchmark harness pins for this file, and
prints both digests when they disagree.

A mismatch is a warning rather than a hard failure only so that a mirror
difference cannot stop a class mid-session; it can equally mean a truncated
or an altered archive, so look into one before trusting the run. The
WAV-file-count check above is a weak substitute: it catches a wrong or
incomplete extraction, not a changed file.

## Pronunciations and Dictionaries

**What.** A pronunciation dictionary, also called a lexicon, maps a written
word to the sequence of phones it is made of. One line per pronunciation:

```text
danger  D EY N JH ER
```

A word that is pronounced more than one way gets one entry per variant.

**Why.** The recognizer never models words directly. It models phones, and
the dictionary is what connects the two.

This notebook puts it to two uses. It turns every training transcript into
the phone sequence that training and forced alignment score against the
audio, and it fixes the vocabulary the decoder is allowed to output. A word
with no entry can be neither trained on nor recognized.

**How.** We build ours from the **CMU Pronouncing Dictionary** (the
`cmudict` package), which covers well over 100,000 English words and writes
phones in ARPAbet, a plain-ASCII notation: `cat` is `K AE T`.

CMUdict marks vowel stress with a trailing digit (`AH0`, `AH1`, `AH2`). We
strip it, which leaves an inventory of 39 stress-independent phones plus
`SIL` for silence. Model capacity and configuration returns to the trade-off
that makes.

In [ ]:
import cmudict


def build_dictionary(dest):
    '''Write a stress-stripped pronunciation dictionary from CMUdict.'''
    raw = cmudict.dict()  # {"the": [["DH", "AH0"], ["DH", "IY0"]], ...}
    lines = []
    for word, prons in sorted(raw.items()):
        for i, phones in enumerate(prons):
            key = word if i == 0 else f"{word}({i + 1})"
            stripped = [re.sub(r"\d$", "", p) for p in phones]
            lines.append(f"{key} {' '.join(stripped)}")
    dest.write_text("\n".join(lines) + "\n")
    return dest


DICT = WORK / "cmudict_no_stress.dict"
build_dictionary(DICT)
print(f"Dictionary written: {DICT} ({sum(1 for _ in DICT.open())} pronunciations)")

In [ ]:
def lex(path):
    d = {}
    for line in path.read_text().splitlines():
        if line.strip() and not line.startswith("#"):
            w, *ph = line.split()
            d[w] = ph
    return d


L = lex(DICT)
phones = sorted({p for v in L.values() for p in v} | {"SIL"})
variant = next((w for w in L if "(" in w), None)

print(f"{len(L)} dictionary entries, {len(phones)} phones:")
print(" ".join(phones))
if variant:
    print("\nExample variant pronunciation:", variant, L[variant])

### Pronunciation variation

**What.** One spelling, more than one pronunciation. The dictionary lists
each variant as its own entry: the first is written plain, and the later
ones carry a `(2)`, `(3)` suffix, the convention `pstrain` expects. The
suffix is a variant number and nothing more, so `the` and `the(2)` are the
same word, not two words.

Take "the". Before a consonant it reduces to `DH AH`; before a vowel it is
usually `DH IY`. Both are correct, and which one a speaker used is a fact
about the recording rather than about the sentence:

```text
                       ┌─ DH  AH ─┐
... previous word ─────┤          ├───── next word ...
                       └─ DH  IY ─┘
                            "the"
```

**How to read the figure.** The two branches are the two pronunciations of
"the", and they rejoin before the next word starts.

With the `pstrain` default `training.multipron_training = True`, the
utterance's HMM graph, the hidden Markov model assembled from its words,
really does carry both branches at once. That is what multipron training
means: every listed variant of a word is a live path through the graph.

**Why it matters.** Baum-Welch sums state and occupancy posteriors across
those paths, so training learns from whichever pronunciation best matches
the audio, with no hard pre-selection.

The alternative `linear` behavior picks the first pronunciation listed and
models only that, matching stock SphinxTrain's selection rule.

Sharing evidence across plausible paths helps most where speakers genuinely
differ: dialect, speaking rate, and reduced function words.

Variation is not all one thing, though. The words below make a ladder:

- **"the"** is one word whose pronunciation varies by context.
- **"record"** is one word whose pronunciation varies by part of speech: the
  noun stresses the first syllable, the verb the second.
- **"bass"** is two different words that happen to share a spelling, the
  fish and the instrument.

The cell below prints all three as the dictionaries actually store them.

In [ ]:
# The Arctic benchmark data that ships with pstrain includes the corpus
# dictionary this notebook's own training data was built against, so we can
# read it straight out of the installed package.
from pstrain.benchmarks.arctic import DATA_DIR

LADDER = ("the", "record", "bass")


def entries_for(lines, words=LADDER):
    """The literal dictionary lines whose headword is one of `words`."""
    kept = []
    for line in lines:
        fields = line.split()
        if fields and fields[0].split("(")[0] in words:
            kept.append(line)
    return kept


print("cmudict as shipped, stress digits kept:")
for entry in entries_for(cmudict.dict_string().splitlines()):
    print("   ", entry)

CORPUS_DICT = DATA_DIR / "cmu_arctic_slt.dict"
corpus_entries = entries_for(CORPUS_DICT.read_text().splitlines())
print(f"\n{CORPUS_DICT.name}, the corpus dictionary, stress stripped:")
for entry in corpus_entries:
    print("   ", entry)

covered = {entry.split()[0].split("(")[0] for entry in corpus_entries}
absent = [word for word in LADDER if word not in covered]
print("    absent from this dictionary:", ", ".join(absent) if absent else "none")

That output repays a second look.

The trailing digit on a CMUdict vowel is stress: `AH0` is unstressed, `AH1`
carries primary stress, `AH2` secondary. Stripping it is what turns
`the DH AH0` and `the(2) DH AH1` into the same phone sequence, `DH AH`. The
corpus dictionary keeps both lines, so `the` and `the(2)` are now literally
the same pronunciation, and only `DH IY` remains distinct.

That is exactly the contrast the branch above draws. The stress difference
between the two `AH` variants is not modeled; the vowel difference is.

"record" and "bass" have no entry in the corpus dictionary at all. A corpus
dictionary carries only the words its transcripts actually use, and neither
word occurs in the CMU_ARCTIC prompts.

Both are in CMUdict, which is why the first block lists them, and so both
are in the dictionary this notebook built a couple of cells back.

> **Checkpoint.** Why do we strip stress digits from the phones at all —
> what would we gain, and what would we lose, by keeping `AH0`/`AH1`/`AH2`
> as distinct units?
>
> <details><summary>Answer</summary>
>
> Keeping stress distinctions gives the model more context to work with —
> a stressed and unstressed vowel really do sound different — but it also
> multiplies the number of phone units, so each unit is seen in less
> training data.
>
> With a small corpus like the one this notebook uses, that data-per-unit
> cost usually outweighs the benefit; with a much larger corpus, keeping
> stress can help. This is the same bias/variance-style trade-off you'd
> expect from adding features to any other model with limited training
> examples.
>
> </details>

Finally we read the transcripts CMU_ARCTIC ships with. The raw file
(`etc/txt.done.data`) uses the Festival prompt format,
`( fileid "words..." )`, one line per utterance.

In [ ]:
if IS_FULL:
    rr = re.compile(r'\(\s*(\S+)\s+"(.*)"\s*\)')
    prompts = {
        m.group(1): m.group(2)
        for line in (ARCTIC / "etc/txt.done.data").read_text().splitlines()
        if (m := rr.match(line.strip()))
    }
else:
    # The bundled mini corpus carries plain `fileid words...` transcripts
    # rather than CMU_ARCTIC's Festival prompt file.
    prompts = dict(
        line.split(maxsplit=1)
        for line in (ARCTIC / "transcription.txt").read_text().splitlines()
        if line.strip()
    )
print(f"{len(prompts)} transcribed prompts")
for uid, text in list(prompts.items())[:3]:
    print(" ", uid, "-", text)

# Speech Analysis

## Speech as a signal

**What.** A microphone samples air-pressure changes. CMU_ARCTIC WAVs store
one 16-bit mono sample 16,000 times per second, so a one-second recording is
16,000 numbers.

**Why.** Nothing downstream looks at those numbers one at a time. A single
sample carries no phonetic information; what carries it is the pattern
across a stretch of them.

**How.** We cut the signal into overlapping **frames**: short slices, about
25 ms each, treated as if the signal held still inside them, because speech
is roughly stationary over that span. A 10 ms step between frame starts
means consecutive frames overlap by more than half, which preserves timing
detail that non-overlapping frames would smear.

We load audio with `librosa.load`, which decodes the file and returns a
float32 array scaled to $[-1, 1]$ together with the sample rate — the same
16-bit-PCM-to-float conversion we'd otherwise do by hand.

One thing to watch for: `librosa.load` **resamples to 22,050 Hz by
default**. CMU_ARCTIC is natively 16,000 Hz, and `pstrain`'s features assume
that rate too, so we always pass `sr=None` to keep the file's native rate
instead.

In [ ]:
from IPython.display import Audio, display

# Different sections listen to different sentences, so that each figure is a
# new thing to hear rather than the same one again. Fixed indices into the
# sorted file list, so every run picks the same two.
corpus_uids = sorted(path.stem for path in WAV_DIR.glob("*.wav"))
EXAMPLE_A = corpus_uids[0]
EXAMPLE_B = corpus_uids[199 % len(corpus_uids)]

example_wav = WAV_DIR / f"{EXAMPLE_A}.wav"
x, sr = librosa.load(example_wav, sr=None, mono=True)
t = np.arange(len(x)) / sr

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(t, x, lw=0.5)
ax[0].set(title=f"Whole utterance ({EXAMPLE_A})", xlabel="time (s)", ylabel="amplitude")
a = int(0.35 * sr)
z = x[a : a + int(0.025 * sr)]
ax[1].plot(np.arange(len(z)) / sr * 1000, z)
ax[1].set(title="A 25 ms frame", xlabel="time (ms)")
plt.tight_layout()
plt.show()

print(EXAMPLE_A, "|", prompts.get(EXAMPLE_A, ""))
print(example_wav.name, "| sample rate:", sr, "| duration (s):", round(len(x) / sr, 2))

# Hear the signal the plots describe. Every figure below comes with a player
# for the sentence it shows, so a picture can always be checked against what
# it sounds like.
display(Audio(x, rate=sr))

**How to read the figure.** On the left is the whole utterance,
`arctic_a0001`, "Author of the danger trail, Philip Steels, etc." The
envelope alone already shows the sentence's structure: loud voiced stretches
for the vowels, near-silent gaps at the pauses and at stop closures.

On the right is a single 25 ms frame taken from the middle of it. That is
the unit everything downstream works on. Inside one frame the signal looks
almost periodic, and it is that repeating shape, not its position in the
sentence, that carries the phone's identity.

> **Try it yourself.** What you will learn: how much of a sentence you can
> read off a waveform before computing anything at all.
>
> What to change: `EXAMPLE_A` in the cell above, then re-run that cell.
>
> Where to find the inputs: the cell below ranks every corpus sentence by
> the share of stop consonants (`P T K B D G`) in its dictionary
> pronunciation and prints the extremes at both ends. A stop is a closure
> followed by a burst, so stops appear in a waveform as short near-silences
> punched into the envelope.
>
> What to expect: set `EXAMPLE_A = "arctic_a0387"` ("Bob, growing disgusted,
> turned back suddenly and attempted to pass Mab.", the stop-heaviest
> sentence in the corpus) and compare it with `EXAMPLE_A = "arctic_a0516"`
> ("Also, there was awe in their faces.", which has no stops at all). The
> first is visibly chopped into pieces by those closures; the second runs as
> one continuous band of energy.

In [ ]:
STOPS = {"P", "T", "K", "B", "D", "G"}
# Silence and sentence markers are not speech sounds, so they must not count
# toward either side of the ratio.
NON_SPEECH = {"SIL", "<s>", "</s>", "<sil>"}


def stop_share(text):
    """Share of stop consonants among a sentence's dictionary phones.

    Project setup does this normalization properly; two lines are enough to
    rank sentences.
    """
    words = [w.strip("'") for w in re.sub(r"[^\w\s']", " ", text.lower()).split()]
    words = [w for w in words if w]
    if not words or any(w not in L for w in words):
        return None
    pronunciation = [p for w in words for p in L[w] if p not in NON_SPEECH]
    if len(pronunciation) < 20:
        return None
    return sum(p in STOPS for p in pronunciation) / len(pronunciation)


ranked = sorted(
    (share, uid)
    for uid, text in prompts.items()
    if (share := stop_share(text)) is not None
)
print(f"{len(ranked)} sentences ranked by stop-consonant share\n")
print("most stops:")
for share, uid in ranked[: -4 : -1]:
    print(f"  {share:.2f}  {uid}  {prompts[uid]}")
print("\nfewest stops:")
for share, uid in ranked[:3]:
    print(f"  {share:.2f}  {uid}  {prompts[uid]}")

## From waveform to spectrum

**What.** The waveform is amplitude over time, and the transform turns each
short window of it into a spectrum, a frequency representation carrying the
same information as the temporal one.

**Why.** What separates one phone from another is mostly where the energy
sits in frequency: the formant pattern of a vowel, the high hiss of a
fricative, the burst of a stop.

In the time domain those live in the fine structure of a wiggle and are
nearly impossible to measure. In the frequency domain they are simply where
the peaks are.

**How.** The short-time Fourier transform (STFT) applies a window to each
frame, takes a 512-point real FFT, and advances 160 samples (10 ms) between
frames. The window is a choice, Hann and Hamming being the usual ones, and
`librosa.stft` takes it as `window=`; the call below uses Hann.

Stacking the resulting spectra shows *when* each frequency's energy occurs:
a spectrogram. `librosa.stft` does exactly this. We fix the window and hop
lengths, in samples at our known 16 kHz rate, once here and reuse them for
every spectrogram in this notebook.

**A note on phase.** An FFT returns a magnitude and a phase for every
frequency bin, and everything downstream in this notebook uses the magnitude
only.

Conventional speech features discard phase because the short-time magnitude
carries most of what these models use, and magnitude is what the mel filters
and the MFCCs go on to operate on.

Phase is not inaudible. It shapes the waveform, and a large enough change to
it can be heard. It is dropped because it costs this kind of model little,
not because it holds nothing.

This is the first step in the chain that throws information away, and it is
worth marking as such. From here on the features are a lossy summary of the
audio, not another encoding of it.

In [ ]:
N_FFT = 512
WIN_LENGTH = round(0.025625 * sr)  # ~410 samples (~25.6 ms)
HOP_LENGTH = round(0.01 * sr)      # 160 samples (10 ms)


def power_spectrogram(y, sr=sr, n_fft=N_FFT, win_length=WIN_LENGTH, hop_length=HOP_LENGTH):
    """Power spectrogram (freq bins x frames) via librosa's STFT."""
    S = librosa.stft(y, n_fft=n_fft, win_length=win_length, hop_length=hop_length, window="hann")
    return np.abs(S) ** 2


# A second sentence, so the spectrum is not the waveform section's audio
# again. The mel and MFCC figures stay with this one, so the whole front end
# can be followed on a single utterance.
xb, srb = librosa.load(WAV_DIR / f"{EXAMPLE_B}.wav", sr=None, mono=True)
P = power_spectrogram(xb, sr=srb)

fig, ax = plt.subplots(figsize=(11, 4))
img = librosa.display.specshow(
    librosa.power_to_db(P, ref=np.max),
    sr=srb, hop_length=HOP_LENGTH, n_fft=N_FFT, x_axis="time", y_axis="hz",
    cmap="magma", ax=ax,
)
ax.set_ylim(0, 8000)
ax.set(title=f"librosa STFT power spectrogram ({EXAMPLE_B})", ylabel="frequency (Hz)")
fig.colorbar(img, ax=ax, format="%+2.0f dB")
plt.show()

print(EXAMPLE_B, "|", prompts.get(EXAMPLE_B, ""))
display(Audio(xb, rate=srb))

**How to read the figure.** Time runs left to right, frequency bottom to
top, and color is energy in decibels relative to the loudest point in the
utterance. The sentence is `arctic_a0200`, "He leapt again, and the club
caught him once more."

The fine horizontal stripes stacked through a voiced stretch are pitch
harmonics, evenly spaced multiples of the speaker's fundamental frequency.

The broad dark bands running across them are formants, the resonances of the
vocal tract, and those are what tell one vowel from another. The narrow
vertical white gaps are stop closures, and the diffuse high-frequency clouds
are fricatives.

The mel filterbank below is built to keep the second of those and discard
the first.

> **Checkpoint.** The window is about 25 ms and the hop is 10 ms, so
> consecutive frames overlap by more than half. Why not just use
> non-overlapping 25 ms frames — what would we lose?
>
> <details><summary>Answer</summary>
>
> Non-overlapping frames would only update our picture of the signal every
> 25 ms, which is coarse relative to how quickly phones change (some last
> only 40–60 ms total).
>
> The 10 ms hop gives roughly 2–3 frames per phone even in fast speech,
> which is what lets the HMM's states actually track sub-phone timing rather
> than seeing each phone as a single blurred observation.
>
> </details>

> **Try it yourself.** What you will learn: the time/frequency resolution
> trade-off that every choice of window length forces on you.
>
> What to change: the `win_length` argument of the `power_spectrogram(...)`
> call in the cell above. Use
> `power_spectrogram(xb, sr=srb, win_length=round(0.05 * sr))` for a 50 ms
> window and re-run the cell.
>
> What to expect: finer frequency detail, so the harmonic stripes become
> narrow enough to count, and blurrier timing, so the vertical edges at stop
> closures smear. A short window, `round(0.010 * sr)`, does the opposite:
> crisp edges, and harmonics that merge into a single band.

## From spectra to MFCCs

**What.** An MFCC, a mel-frequency cepstral coefficient, is one number in a
short vector that summarizes the shape of a frame's spectrum. Thirteen of
them describe a 25 ms window well enough to train an acoustic model on.

**Why.** Raw spectral power is a poor feature: high-dimensional, strongly
correlated between neighboring bins, and not perceptually scaled.

Worse, the part we want and the part we do not are mixed together in it. A
speech signal is a source convolved with a filter: glottal pulses, which
carry pitch and identify the speaker, shaped by the vocal tract, whose
resonances carry the phone. We want those two separated.

**How.** Convolution in time is multiplication in frequency, so taking the
spectrum already turns the convolution into a product. Take the **log** and
the product becomes a sum.

Run an inverse transform over that log spectrum and the two summed parts
land at different ends of the new axis: the slowly varying vocal-tract
envelope low down, the fast ripple of the pitch harmonics high up.

Keep the low coefficients, drop the rest, and source and filter are
approximately separated. Only approximately: low quefrency mainly carries
the smooth envelope and higher quefrency the harmonic fine structure, but
the two overlap rather than occupying disjoint ranges.

That is a **homomorphic** transform, and the log is the homomorphic step.
Turning a product into a sum is what lets a linear operation pull apart two
signals that were multiplied together.

The chain is spectrum → mel filterbank → log → inverse transform. The
result has a name.

**Cepstrum.** The real cepstrum of a signal $x$ is the inverse Fourier
transform of its log-magnitude spectrum:

$$c_x = \mathcal{F}^{-1}\{\log|\mathcal{F}\{x\}|\}$$

with $\mathcal{F}$ the Fourier transform. A log-power convention is equally
common and differs from this one only by a scale.

Bogert, Healy and Tukey coined the word in 1963 by flipping the first
syllable of "spectrum", because this is a transform applied to a spectrum
rather than to a waveform. The same joke gives "quefrency" for the
cepstrum's axis and "liftering" for filtering along it. The plural is
cepstra and the adjective is cepstral.

An MFCC follows the same idea with the mel step in between, and is related
to the classical cepstrum rather than identical to it. It applies a discrete
cosine transform (DCT) to the log energies of a mel filterbank: a
cosine-basis transform that compacts those energies into the first few
coefficients and largely decorrelates them.

What this buys an HMM/GMM system:

- The DCT leaves the coefficients close to uncorrelated, which is what lets
  each Gaussian use a **diagonal** covariance instead of a full one. That is
  what made large diagonal-covariance GMM models practical to train and
  evaluate.
- Thirteen numbers carry the envelope of a 25 ms window, so a second of
  speech becomes 100 short vectors rather than 100 full spectra.

The full production chain `pstrain` uses is:

pre-emphasis (0.97) → 25.6 ms frames / 10 ms step → 512-point FFT →
25 mel filters (130–6800 Hz) → log → 13-point DCT → lifter (22) →
batch cepstral mean normalization → deltas and delta-deltas
→ **39-dimensional feature vectors**.

Cepstral mean normalization (CMN) subtracts the average cepstrum, over the
utterance or the batch, from every frame. That removes whatever stayed
constant through the recording: a channel, a microphone, a good deal of a
speaker.

The deltas and delta-deltas are the first and second differences between
neighboring frames. They are there because an HMM state scores one frame at
a time and would otherwise see no motion at all.

The visualization below stops at log-mel energies, which is already enough
to see why the transform helps.

We compute the actual MFCCs, DCT, liftering, and CMN included, with
`librosa.feature.mfcc` in Features and configuration, right next to
`pstrain`'s own real extracted training features, so the two can be compared
directly.

In [ ]:
N_MELS, FMIN, FMAX = 25, 130, 6800

mel_fb = librosa.filters.mel(sr=srb, n_fft=N_FFT, n_mels=N_MELS, fmin=FMIN, fmax=FMAX)
mel_power = librosa.feature.melspectrogram(
    y=xb, sr=srb, n_fft=N_FFT, win_length=WIN_LENGTH, hop_length=HOP_LENGTH,
    n_mels=N_MELS, fmin=FMIN, fmax=FMAX, window="hann",
)
log_mel = librosa.power_to_db(mel_power, ref=np.max)

freqs = librosa.fft_frequencies(sr=srb, n_fft=N_FFT)
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(freqs, mel_fb.T)
ax[0].set(xlim=(0, 8000), title="Mel filterbank (librosa.filters.mel)", xlabel="Hz")
img = librosa.display.specshow(
    log_mel, sr=srb, hop_length=HOP_LENGTH, x_axis="time", cmap="magma", ax=ax[1]
)
ax[1].set(title=f"Log-mel spectrogram ({EXAMPLE_B})")
plt.tight_layout()
plt.show()

display(Audio(xb, rate=srb))

**How to read the figure.** On the left, each triangle is one mel filter: a
weighting over FFT bins that sums the energy falling underneath it. They are
narrow and closely packed at low frequencies and wide and sparse at high
ones, which is the mel scale.

On the right is the same sentence as the spectrogram above, after those
filters are applied: 25 rows instead of 257. The harmonic stripes are gone,
because each filter averaged across them, and what survives is the
formant envelope. That is the discarding the mel step is for.

> **Checkpoint.** Look at the mel filterbank on the left: the filters are
> narrow and closely spaced at low frequencies and wide and sparse at high
> frequencies. Why?
>
> <details><summary>Answer</summary>
>
> The mel scale approximates how human hearing perceives pitch: we're much
> more sensitive to differences at low frequencies (where most phonetic
> information — vowel formants, voicing — lives) than at high ones.
>
> Packing more filters into the low end spends the feature budget where it
> perceptually and phonetically matters most, at the cost of frequency
> resolution up high where it matters least.
>
> </details>

> **Try it yourself.** What you will learn: where the useful detail in a
> filterbank stops, and why more filters is not automatically better.
>
> What to change: `N_MELS` at the top of the cell above, from 25 to 40. Both
> `librosa.filters.mel` and `librosa.feature.melspectrogram` read it as
> `n_mels`, so the one edit changes both panels.
>
> What to expect: more, narrower triangles on the left, and a smoother
> log-mel image on the right with visibly more vertical detail. What it will
> not change is the model: `pstrain`'s production chain keeps only the first
> 13 DCT coefficients afterward, so past a point extra mel filters buy
> computation rather than information.

# Project setup

We now turn the raw prompts into a training-ready transcript file: convert
each simple `fileid words…` prompt into the form `pstrain` expects, filter
out any utterance whose words aren't in our dictionary, and select a working
subset (`N_UTTS`, set in Configuration).

A smaller subset trains and iterates faster while you're developing; a
larger one gives a better model.

We lowercase, strip punctuation except internal apostrophes, and trim edge
apostrophes to match the lowercase-only dictionary.

In [ ]:
def norm(t):
    t = re.sub(r"[^\w\s']", " ", t.lower())
    return " ".join(q.strip("'") for q in t.split() if q.strip("'"))


normalized = {u: norm(t) for u, t in prompts.items()}
in_vocabulary = {
    u: t
    for u, t in normalized.items()
    if t
    and all(w in L for w in t.split())
    and (WAV_DIR / f"{u}.wav").exists()
    and u not in EXCLUDE_UTTERANCES
}
selected = dict(sorted(in_vocabulary.items())[:N_UTTS])
if EXCLUDE_UTTERANCES:
    print(f"Excluding {len(EXCLUDE_UTTERANCES)} utterance(s) per EXCLUDE_UTTERANCES: "
          f"{EXCLUDE_UTTERANCES}")

TRANS = WORK / "all.transcription"
TRANS.write_text("".join(f"{u} {t}\n" for u, t in selected.items()))

print(
    f"{len(prompts)} prompts -> {len(in_vocabulary)} fully in-vocabulary "
    f"-> {len(selected)} selected (N_UTTS={N_UTTS})"
)

# Three more sentences, none of them the two the sections above listened to.
gallery = [(u, t) for u, t in selected.items() if u not in (EXAMPLE_A, EXAMPLE_B)][:3]
fig, axs = plt.subplots(len(gallery), 1, figsize=(11, 2 * len(gallery)))
examples = []
for ax, (u, t) in zip(np.atleast_1d(axs), gallery):
    y, r = librosa.load(WAV_DIR / f"{u}.wav", sr=None, mono=True)
    examples.append((u, t, y, r))
    q = power_spectrogram(y, sr=r)
    librosa.display.specshow(
        librosa.power_to_db(q, ref=np.max),
        sr=r, hop_length=HOP_LENGTH, x_axis="time", y_axis="hz", cmap="magma", ax=ax,
    )
    ax.set_ylim(0, 8000)
    ax.set_title(f"{u}: {t}", fontsize=9)
plt.tight_layout()
plt.show()

# One player per spectrogram above, in the same order: reading a transcript
# while hearing it is what makes the spectrogram legible.
for u, t, y, r in examples:
    print(f"{u}: {t}")
    display(Audio(y, rate=r))

**How to read the figure.** Each row shows an utterance with its transcript
above and its player below.

Read a transcript while looking at its spectrogram. The sentences differ in
length, in how much silence they carry, and in where their energy sits, and
none of that is visible in a file listing.

That spread is the variation training has to absorb, which is why the number
of utterances matters as much as the model's size.

In [ ]:
bad = []
for u, t in selected.items():
    with wave.open(str(WAV_DIR / f"{u}.wav"), "rb") as w:
        if (w.getframerate(), w.getnchannels(), w.getsampwidth()) != (16000, 1, 2):
            bad.append(u)
    if any(q not in L for q in t.split()):
        bad.append(u)
assert not bad
print(f"OK: {len(selected)} WAVs are 16 kHz/mono/16-bit; all words are in-vocabulary")

Context-dependent (`cd-*`) training clusters neighboring-phone effects using
a decision tree (see the Training section), which needs enough occurrences of
each phone in context to build reliably.

With very few utterances that clustering can become unstable, so we fall back
to the simpler context-independent target `ci-1g` when the selection is too
small. Raise `N_UTTS` in Configuration for a real `cd-*` experiment.

In [ ]:
if len(selected) < 100 and TARGET.startswith("cd-"):
    print(
        f"Only {len(selected)} utterances selected; context-dependent "
        f"training ({TARGET}) needs more data than that to build reliable "
        "decision trees, so this run falls back to ci-1g. Raise N_UTTS in "
        "the configuration cell for a real cd- experiment."
    )
    TARGET_RUN = "ci-1g"
else:
    TARGET_RUN = TARGET

print("Training target for this run:", TARGET_RUN)

In [ ]:
import shutil

from pstrain.api import setup_project

# Rebuild the project scaffold from *this run's* TRANS every time, instead
# of layering on top of whatever an earlier run with a different
# N_UTTS/EXCLUDE_UTTERANCES left behind. That earlier layering is exactly
# how an excluded utterance can keep training anyway: its audio link and
# cached features from a previous run are still sitting on disk, and
# pstrain's mtime-driven pipeline (by design) reuses anything it finds
# rather than recomputing it.
#
# We remove the whole project directory ourselves rather than leaning on
# setup_project(..., clobber=True) alone: with link_audio=True, "audio" is
# a symlink to WAV_DIR, and pstrain's own clobber path can trip over a
# symlink already sitting where it wants to (re)create one -- "Refusing to
# write through destination symlink". shutil.rmtree() only removes the
# symlink *entry* here (it does not follow it), so this is safe: WAV_DIR
# and its real WAV files are untouched. This is cheap -- it's just file
# linking, not retraining -- so we always do it.
PROJECT = WORK / "slt_project"
if PROJECT.exists():
    shutil.rmtree(PROJECT)
setup = setup_project(PROJECT, TRANS, WAV_DIR, DICT, link_audio=True, clobber=True)
print("project:", PROJECT)
print(setup)
for q in sorted(PROJECT.rglob("*")):
    if q.is_file() or q.is_symlink():
        print(" ", q.relative_to(PROJECT))

In [ ]:
from pstrain.api import parse_transcription_file
from pstrain.api.pipeline import PipelineContext, build_pipeline

ctx = PipelineContext.from_config(
    PROJECT,
    experiment="default",
    config_name="default",
    cli_overrides={"runner": {"jobs": JOBS}},
)
pipeline = build_pipeline(ctx)
assert pipeline.run("split", jobs=JOBS) == 0

ETC = PROJECT / "experiments/default/etc"
train_transcripts = parse_transcription_file(ETC / "train.transcription")
test_transcripts = parse_transcription_file(ETC / "test.transcription")

counts = [len(train_transcripts), len(test_transcripts)]
fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.pie(
    counts,
    labels=[f"train\n{counts[0]} utterances", f"test\n{counts[1]} utterances"],
    autopct=lambda pct: f"{pct:.0f}%",
    startangle=90,
    counterclock=False,
    colors=["#4c72b0", "#dd8452"],
)
ax.set_title("Train/test split (seed 42)")
plt.show()
print("train:", counts[0], "| test:", counts[1])

**How to read the figure.** The split is by utterance, not by time or by
speaker. It is drawn once with a fixed seed, so the same utterances land on
the same side of the line on every run.

The test wedge is held out of training completely. No frame from it reaches
Baum-Welch, which is what makes the error rate reported later a measurement
rather than a memory.

# Features and configuration

The production feature chain ends in normalized 39-dimensional cepstra.
`feat.params` persists the entire front-end contract beside the model, so
decoding later reproduces training features exactly.

In [ ]:
from pstrain.api import resolve_config

cfg = resolve_config(PROJECT, profile_name="default").as_dict()
feature_fields = [
    "samprate", "ncep", "nfilt", "nfft", "frate", "wlen",
    "feat_type", "alpha", "lifter",
]
for field in feature_fields:
    print(f"features.{field}: {cfg['features'][field]}")

for field in ("n_state", "n_senones"):
    print(f"training.{field}: {cfg['training'][field]}")

for family in ("ci", "untied", "tied"):
    print(f"training.{family}: {cfg['training'][family]}")

In [ ]:
# Feature extraction runs over every utterance in the linked audio, not just
# the N_UTTS selected ones: pstrain plans it from the audio directory, before
# the train/test split exists. For CMU_ARCTIC that is ~1,132 MFCC files and
# roughly 20 seconds, paid once and then cached.
assert pipeline.run("features", jobs=JOBS) == 0
fp = PROJECT / "shared/features/default/feat.params"
print(fp.read_text())

# The same cepstral chain by hand, on the sentence the spectrum and log-mel
# figures used: STFT -> mel filterbank -> log -> DCT -> lifter, matching
# pstrain's front-end settings printed above (13 coefficients, lifter 22) as
# closely as librosa's API allows, plus per-utterance CMN.
mfcc = librosa.feature.mfcc(
    y=xb, sr=srb, n_mfcc=13, n_fft=N_FFT, win_length=WIN_LENGTH, hop_length=HOP_LENGTH,
    n_mels=N_MELS, fmin=FMIN, fmax=FMAX, window="hann", lifter=22,
)
mfcc -= mfcc.mean(axis=1, keepdims=True)  # cepstral mean normalization

fig, ax = plt.subplots(figsize=(11, 3))
img = librosa.display.specshow(mfcc, sr=srb, hop_length=HOP_LENGTH, x_axis="time", cmap="coolwarm", ax=ax)
ax.set(
    title=f"librosa MFCCs, mean-normalized ({EXAMPLE_B})",
    ylabel="coefficient",
)
fig.colorbar(img, ax=ax)
plt.show()

# The same utterance these coefficients describe.
display(Audio(xb, rate=srb))

**How to read the figure.** Each column is one frame. Each row is one
cepstral coefficient, index 0 at the bottom.

Low-index coefficients describe broad structure in the spectral envelope
across the mel bands: the overall tilt first, then progressively coarser
shape. Higher-index coefficients describe finer detail across those same
bands.

The vertical axis is coefficient index, not acoustic frequency. It is
analogous to quefrency, but this is a DCT of mel-band log energies rather
than a full cepstrum, so no row corresponds to a frequency you could name.

The picture is centered on zero because mean normalization subtracted this
utterance's average cepstrum from every frame. Neighboring rows are close to
uncorrelated, which is the property a diagonal-covariance Gaussian needs.

# The model

**What.** Each phone is modeled as a small **hidden Markov model** (HMM): a
left-to-right chain of three emitting states, each of which scores a feature
frame with a **Gaussian mixture model** (GMM).

**Why.** A phone is not one sound. It has an onset, a steady middle, and a
transition out of it, and those regions look different from each other.
Giving a phone three states lets the model say which of them a frame belongs
to.

The division of labor is clean. The HMM models *time*, meaning which
acoustic region we are in and for how long. The GMM models *feature space*,
meaning what that region's frames look like.

**How.** Each state has a self-loop, which consumes another frame without
moving on, and a forward arc to the next state. A slow talker and a fast one
are then the same model traversed with different numbers of loops, which is
how one topology absorbs a large amount of duration variation.

`pstrain` starts those arcs at 0.75 for the self-loop and 0.25 for the step
forward, and training re-estimates them.

Each state scores 39-D feature frames with a diagonal-covariance Gaussian
mixture. For a $D$-dimensional diagonal covariance
$\Sigma=\mathrm{diag}(\sigma_1^2,\dots,\sigma_D^2)$, the log density of a
single Gaussian is

$$\log \mathcal{N}(x;\mu,\Sigma)=-\frac{D}{2}\log(2\pi)-\frac{1}{2}\sum_{d=1}^{D} \log \sigma_d^2-\frac{1}{2}\sum_{d=1}^{D} \frac{(x_d-\mu_d)^2}{\sigma_d^2}$$

with $D=39$ for the real features above.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 2.8))

RADIUS = 0.42
STATE_X = [1.8, 3.8, 5.8]
ENTRY_X, EXIT_X = 0.0, 7.6


def forward_arc(x0, r0, x1, r1, label=None):
    ax.annotate(
        "", xy=(x1 - r1, 0), xytext=(x0 + r0, 0),
        arrowprops=dict(arrowstyle="-|>", linewidth=1.5, color="#22456e"),
    )
    if label:
        ax.text((x0 + x1) / 2, 0.10, label, ha="center", va="bottom", fontsize=9)


for px, name in ((ENTRY_X, "entry"), (EXIT_X, "exit")):
    ax.add_patch(plt.Circle((px, 0), 0.24, facecolor="white", edgecolor="0.45",
                            linestyle="--", linewidth=1.3, zorder=3))
    ax.text(px, -0.60, name, ha="center", va="top", fontsize=8, color="0.35")

for index, px in enumerate(STATE_X, start=1):
    ax.add_patch(plt.Circle((px, 0), RADIUS, facecolor="#dce6f2", edgecolor="#22456e",
                            linewidth=1.7, zorder=3))
    ax.text(px, 0, f"s{index}", ha="center", va="center", fontsize=12, zorder=4)
    # Self-loop: an arc that leaves the top of the state and comes back.
    ax.annotate(
        "", xy=(px + 0.20, 0.37), xytext=(px - 0.20, 0.37),
        arrowprops=dict(arrowstyle="-|>", linewidth=1.5, color="#22456e",
                        connectionstyle="arc3,rad=-1.6"),
    )
    ax.text(px, 0.80, "0.75", ha="center", va="bottom", fontsize=9)

forward_arc(ENTRY_X, 0.24, STATE_X[0], RADIUS)
forward_arc(STATE_X[0], RADIUS, STATE_X[1], RADIUS, "0.25")
forward_arc(STATE_X[1], RADIUS, STATE_X[2], RADIUS, "0.25")
forward_arc(STATE_X[2], RADIUS, EXIT_X, 0.24, "0.25")

ax.set_xlim(-0.8, 8.4)
ax.set_ylim(-1.0, 1.2)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("One phone model: three emitting states, with entry and exit", fontsize=11)
plt.tight_layout()
plt.show()

**How to read the figure.** The three shaded circles are the emitting
states, the ones that actually score a frame against a Gaussian mixture.
Each carries a self-loop, which consumes another frame without advancing,
and a forward arc to its successor.

The 0.75 / 0.25 pair is the model's prior on duration before training has
seen anything. Staying is three times as likely as moving on, which puts the
expected stay at about four frames, or 40 ms, close to a short phone's real
duration.

The dashed circles are the non-emitting entry and exit pseudo-states. They
score nothing and consume no time. They exist so models can be chained: the
exit of one phone is the entry of the next, and a word is its phones glued
together at those joints.

Phone models chain into words and words into the search network the decoder
walks; cross-word context and the language model come in Alignment and
decoding.

The demo below is not derived from real speech features. It is a toy with
$D=2$, built to make "soft assignment" concrete before we let `pstrain` fit
real 39-dimensional Gaussians: two Gaussians, a scatter of points, and each
point colored by how much of it each component claims.

In [ ]:
np.random.seed(0)

points = np.r_[
    np.random.randn(80, 2) * [0.5, 0.8] + [-1, 0],
    np.random.randn(70, 2) * [0.6, 0.4] + [1.2, 1],
]
means = np.array([[-1, 0], [1.2, 1]])
variances = np.array([[0.25, 0.64], [0.36, 0.16]])


def diagonal_gaussian_pdf(values, mean, variance):
    exponent = -0.5 * np.sum((values - mean) ** 2 / variance, axis=1)
    normalizer = np.sqrt((2 * np.pi) ** 2 * np.prod(variance))
    return np.exp(exponent) / normalizer


# Normalize weighted likelihoods to obtain posterior component responsibilities.
weighted_likelihoods = np.c_[
    0.55 * diagonal_gaussian_pdf(points, means[0], variances[0]),
    0.45 * diagonal_gaussian_pdf(points, means[1], variances[1]),
]
responsibilities = weighted_likelihoods / weighted_likelihoods.sum(axis=1, keepdims=True)

plt.scatter(*points.T, c=responsibilities[:, 1], cmap="coolwarm")
plt.scatter(*means.T, color="black", marker="x", s=100)
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("Diagonal-GMM soft assignments")
plt.colorbar(label="responsibility for component 2")
plt.show()

**How to read the figure.** Each dot is one two-dimensional point, colored by
the responsibility the second component takes for it: blue where the first
component explains it, red where the second does. The black crosses are the
two component means.

The band between them is the part to look at. Color there changes gradually
rather than switching, and every point in it contributes a fraction of its
evidence to both components when the parameters are re-estimated.

> **Checkpoint.** Points near the boundary between the two clusters get
> responsibilities close to 0.5 rather than being forced to pick a side.
> Why is that "soft" assignment useful for training, rather than just
> hard-assigning each point to its nearer mean (like k-means)?
>
> <details><summary>Answer</summary>
>
> Speech frames genuinely are ambiguous at phone boundaries and during
> coarticulation — a frame really can be "a bit of both" acoustic regions.
>
> Soft assignment lets every point contribute a fractional amount of
> evidence to both components in proportion to how well each explains it,
> which is exactly what the forward-backward algorithm does across HMM
> states in the Training section.
>
> Hard assignment would throw away that gradation and make training more
> sensitive to where an arbitrary boundary falls.
>
> </details>

> **Try it yourself.** What you will learn: that soft assignment is not a
> two-way idea; it is what a mixture of any size does.
>
> What to change: in the cell above, add a third component. Pick a mean and
> a variance, add a third column to `weighted_likelihoods` (the weights need
> not sum to 1 before normalization, but keeping them near it is tidier),
> and color the scatter by `responsibilities[:, 2]`.
>
> What to expect: the single soft boundary becomes two, and points that were
> confidently assigned before now split their evidence three ways wherever
> the new component reaches them.

## Model families

The families, in increasing order of how much context they model:

- **`ci-1g` … `ci-8g`**: monophone models. Context-independent, meaning one
  HMM per phone whatever its neighbors. In `pstrain`'s CI targets each
  monophone state keeps its own distribution; tying is a separate choice,
  and it is the CD stages that make it.
- **`cd-untied`**: every observed triphone, meaning a phone taken together
  with its immediate left and right neighbors, gets its own states. The same
  few dozen phones become thousands of triphones, most of them seen a
  handful of times, and there is no longer enough data to estimate a
  separate distribution for each. It is a useful intermediate step and
  rarely used directly.
- **`cd-1g` … `cd-32g`**: decision-tree-tied triphone states (see the
  Training section), which share statistical evidence across similar
  contexts. Tying is the answer to the problem the untied stage creates.

The number is the density, the count of Gaussians per state. `cd-8g` is the
toolkit default; `cd-1g`, this notebook's default, is the fast
full-context-dependent path, good for iterating quickly.

Terms that are easy to run together:

- a **phoneme** is an abstract category: the unit that distinguishes one word
  from another in a language, so that swapping it changes which word was
  said;
- a **phone** is a concrete realization of one, an actual stretch of sound
  somebody produced. The ARPAbet symbols in our dictionary are the units we
  model, and recognition work calls them phones by convention even though
  they sit closer to phonemes;
- an HMM **state** is a temporal phase inside a phone model;
- a **senone** is a shared tied-state distribution, pooling several triphone
  states together;
- a **Gaussian** is one mixture component inside a senone's GMM.

A dry run doesn't train anything. It asks `pstrain` which stages it would
run, in what order, and why each one is due.

The plan prints one tab-separated row per stage, in the same shape as the
progress rows the run itself prints, so a plan reads as a prediction of that
run. A fan-out reports once for the whole group rather than once per
utterance, so the shape of the build stays readable even though this plan
holds over a thousand tasks:

```text
index	stage	tasks	status	description
1	provenance:split	1	not built yet	Record effective split configuration
2	split	1	upstream 'provenance:split' will run	Partition all.transcription into train/test fileids + transcripts
3	provenance:features	1	not built yet	Record effective features configuration
4-1135	features	1132	stale
1136	provenance:training	1	not built yet	Record effective training configuration
1137	flat	1	upstream 'split' will run	Initialize flat (uniform) acoustic model
```

`index` is a position in the plan, and a range is a collapsed fan-out.
`status` says why a stage is due: "not built yet" when its output does not
exist, "stale" when the output is older than its inputs, and
"upstream '...' will run" for a stage that is due only because something it
depends on is.

"Not built yet" is the normal state for a fresh project, since nothing has
trained and so no stage's output exists. It stops being true once you run
the training cell below.

The `provenance:*` entries are bookkeeping steps that record the exact
configuration each stage ran with, for caching and reproducibility. Pass
`verbose=True` to list every task individually.

In [ ]:
assert pipeline.run(TARGET_RUN, dry_run=True, jobs=JOBS) == 0

# The stages the plan just listed, with the dependencies the plan reports.
# `split` and `features` are independent roots: neither waits for the other,
# and the build joins them at the first training stage.
positions = {
    "split": (0.0, 0.95),
    "features": (0.0, -0.95),
    "flat": (1.9, 0.0),
    "ci-1g": (3.7, 0.0),
}
edges = [("split", "flat"), ("features", "flat"), ("flat", "ci-1g")]
# Roots: nothing else drawn here has to run before them.
ROOTS = {"split", "features", "alltriphones-mdef"}
if TARGET_RUN != "ci-1g":
    positions.update({
        "questions": (5.5, 1.45),
        "cd-untied-init": (5.5, 0.0),
        "cd-untied": (7.6, 0.0),
        "trees": (9.5, 1.45),
        "prune-trees": (11.4, 1.45),
        "cd-1g-init": (13.4, 0.0),
        "cd-1g": (15.4, 0.0),
        "alltriphones-mdef": (11.0, -1.5),
    })
    edges += [
        ("ci-1g", "questions"), ("ci-1g", "cd-untied-init"),
        ("cd-untied-init", "cd-untied"),
        ("cd-untied", "trees"), ("questions", "trees"),
        ("trees", "prune-trees"),
        ("cd-untied", "cd-1g-init"), ("prune-trees", "cd-1g-init"),
        ("alltriphones-mdef", "cd-1g-init"),
        ("cd-1g-init", "cd-1g"),
    ]


def box_pad(label, dx, dy):
    """How far to hold an arrow off a node, in points, so it stops at the
    box border rather than running under the text. A slanted arrow needs
    less than a level one, hence the reach factor."""
    reach = abs(dx) / max(np.hypot(dx, dy), 1e-9)
    return (2.7 * len(label) + 7) * max(reach, 0.5)


# Width follows the number of stages, so a ci-1g run does not draw four
# boxes across a chart sized for eleven.
span = max(p[0] for p in positions.values())
fig, ax = plt.subplots(figsize=(min(13.5, 3.4 + 0.72 * span), 3.2))
for source, target in edges:
    (sx, sy), (tx, ty) = positions[source], positions[target]
    ax.annotate(
        "", xy=(tx, ty), xytext=(sx, sy),
        arrowprops=dict(arrowstyle="-|>", linewidth=1.3, color="#6d7a89",
                        shrinkA=box_pad(source, tx - sx, ty - sy),
                        shrinkB=box_pad(target, tx - sx, ty - sy)),
    )
for name, (px, py) in positions.items():
    root = name in ROOTS
    ax.text(
        px, py, name, ha="center", va="center", fontsize=9, zorder=3,
        bbox=dict(boxstyle="round,pad=0.35",
                  facecolor="#fbe6d4" if root else "#dce6f2",
                  edgecolor="#c26a35" if root else "#22456e", linewidth=1.4),
    )

ys = [p[1] for p in positions.values()]
ax.set_xlim(-1.2, span + 1.2)
ax.set_ylim(min(ys) - 0.9, max(ys) + 0.9)
ax.axis("off")
ax.set_title(f"Stage dependencies for {TARGET_RUN}", fontsize=11)
plt.tight_layout()
plt.show()

**How to read the figure.** An arrow runs from a stage to the stage that
needs its output, so the arrows point in the order the build must run.

The orange boxes are roots. Nothing else in the figure has to run before
them. `split` partitions the transcripts and `features` extracts the MFCCs,
and neither waits for the other, which is why the plan can list `split`
before a feature fan-out that will take far longer. The build joins them at
`flat`, the first stage that needs both.

`alltriphones-mdef` is a root as well: the architecture's full triphone
inventory, which tying needs in order to map every context, including ones
training never saw, onto a surviving leaf. It joins the build at
`cd-1g-init`.

After that the picture is mostly a chain, with one branch. `questions` comes
off `ci-1g` rather than off the untied model, so the phonetic questions can
be generated while the untied triphones are still training, and the two meet
again at `trees`.

The figure leaves some detail out. The per-utterance feature tasks and the
per-(phone, state) tree tasks are not drawn: each of those boxes stands for
a fan-out that the plan collapses into a single row.

Some edges are implied rather than drawn. Every Baum-Welch stage rereads the
split and the features, not just `flat`, and `cd-1g-init` reads `ci-1g` and
`questions` directly as well as along the paths shown. Drawing each of those
once, on a path that already reaches the stage, keeps the picture readable.

# Training

## Running the pipeline

**HEAVY CELL.** This runs the real `pstrain` training pipeline shown above,
the only cell in this notebook that does substantial computation.

With the default configuration (150 utterances, `cd-1g`) expect roughly
8–15 seconds on a modern many-core laptop; a bundled or very small selection
with `ci-1g` finishes in a couple of seconds.

`pstrain` caches completed stages by input modification time, so re-running
this cell after the first successful pass is fast unless you changed
`N_UTTS`, `TARGET`, or set `FORCE = True` in Configuration.

**If this cell fails**, look for a line starting with `!!` in the output
above the traceback. `pstrain` prints the actual cause there, before the
pipeline reports failure.

> **Note.** Training may drop an utterance it cannot align, and carry on
> without it. The line that says so reads `Final state not reached for
> arctic_a0135 even at a_beam=1e-100; omitting it from this pass and
> continuing`, and the stage ends with a `WARNING: BW training skipped N
> utterance updates in total` line. The per-pass counts live in the
> per-shard Baum-Welch logs under `.pstrain/bw/<stage>/`, and in
> `bw_telemetry.json` beside each stage's model, whose per-pass `accounting`
> block names the utterance and the reason. `training.max_skip_fraction`
> still fails the run once skipping stops being incidental. Set
> `EXCLUDE_UTTERANCES` in Configuration if you want an utterance out of the
> plots and counts entirely, not just out of the model.

In [ ]:
# HEAVY CELL: the real pstrain training pipeline
ctx = PipelineContext.from_config(
    PROJECT,
    experiment="default",
    config_name="default",
    cli_overrides={"runner": {"jobs": JOBS}},
)
pipeline = build_pipeline(ctx)

started = time.perf_counter()
rc = pipeline.run(TARGET_RUN, force=FORCE, jobs=JOBS)
TRAIN_SECONDS = time.perf_counter() - started
if rc != 0:
    raise RuntimeError(
        f"pstrain training failed (exit code {rc}). Look for the line "
        "starting with '!!' above -- that's pstrain's own report of which "
        "stage failed and why. Note that a single utterance too short to "
        "force-align is not the cause: that one is reported and omitted "
        "from the pass, and training continues (see the note above this "
        "cell)."
    )

MODEL_DIR = PROJECT / "shared/models" / TARGET_RUN / "default"
CI_MODEL_DIR = PROJECT / "shared/models/ci-1g/default"
print(f"TRAIN_WALL_SECONDS={TRAIN_SECONDS:.3f}")
print(MODEL_DIR)
print("Equivalent CLI:")
print(f"# pstrain build {TARGET_RUN} --project-dir {PROJECT} -j {JOBS} -c default")

In [ ]:
required = [
    "mdef", "means", "variances", "mixture_weights",
    "transition_matrices", "feat.params",
]
for f in required:
    q = MODEL_DIR / f
    assert q.is_file()
    print(f, q.stat().st_size)

rows = [
    z.split()
    for z in (MODEL_DIR / "mdef").read_text(errors="replace").splitlines()
    if len(z.split()) >= 10 and z.split()[-1] == "N" and not z.startswith("#")
]
print("base lft rt pos attrib tmat s0 s1 s2 N")
for row in rows[:4]:
    print(" ".join(row))

if TARGET_RUN.startswith("cd-"):
    ids = [s for r in rows for s in r[6:9]]
    print(len(ids), "state references,", len(set(ids)), "unique senone IDs")

The pipeline you just ran performed several distinct steps in sequence.
This section peels each one apart with a small, from-scratch demonstration
so the mechanics behind the black box are visible.

## Flat initialization

**What.** Every emitting distribution starts from the same parameters: the
pooled global mean and variance of all the training frames. The topology
starts with a 0.75 self-loop and a 0.25 forward probability, the same pair
the phone-model figure showed.

**Why.** Baum-Welch is iterative re-estimation, and it has to start
somewhere.

Whatever acoustic structure the starting point carries is structure the
model did not learn. The honest start asserts nothing about acoustics: every
state believes the same thing, and every difference between states afterward
came from the data.

**How.** `pstrain` estimates one global Gaussian over all frames in the
corpus and tiles its mean and variance into every state.

Identical emissions do not hand out identical occupancies, though. On the
first pass the states are told apart only through the topology: the
left-to-right ordering, the transition probabilities, the graph the
transcript builds, and how many frames the utterance holds are what shape
the forward-backward occupancies.

Those differences are slight, and re-estimation is what amplifies them.

At this point the HMM topology exists, but no state is acoustically
specialized: every state generates the same distribution as every other.

In [ ]:
np.random.seed(0)
toy = np.r_[np.random.normal(-0.5, 1, (100, 2)), np.random.normal(1, 0.6, (80, 2))]
gm = toy.mean(0)
gv = toy.var(0)
sm = np.tile(gm, (3, 1))

plt.scatter(*toy.T, s=10, alpha=0.3)
plt.scatter(sm[:, 0], sm[:, 1], s=[80, 160, 240], facecolors="none",
            edgecolors=["r", "g", "b"])
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("Flat init: every state shares one Gaussian")
plt.show()
print("global mean:", gm, "| global variance:", gv)

**How to read the figure.** The faint cloud is the pooled training data. The
three open circles are the three states of a phone model at the moment
training starts, drawn at different sizes only so you can tell them apart.

All three sit on the same point, the global mean, because that is the whole
content of a flat start: no state knows anything the others do not.

The later mean updates move these coincident centers apart; variances,
weights and transitions move too.

## Baum-Welch and forward-backward

**What.** Baum-Welch is the algorithm that trains an HMM on audio carrying
no state labels. It is the EM algorithm applied to HMMs, and
forward-backward is the recursion it uses to work out where the states
probably were.

**Why.** Training faces a circular problem. To estimate what a state's
Gaussian looks like you need to know which frames belong to that state; to
know which frames belong to a state you need the state's Gaussian. Neither
is given. The transcript says which phones occur and in what order, never
where one ends and the next begins.

Baum-Welch cuts the circle by refusing to choose. Instead of assigning each
frame to a state, it assigns each frame to *every* state, in proportion to
how probable that state is at that moment given the whole utterance and the
current model.

**How.** For each frame, a forward pass accumulates the probability of
everything heard up to it and a backward pass the probability of everything
after it. Multiplying the two and normalizing gives γ, the posterior
probability of being in each state at each time: a soft assignment that sums
to one across states.

Then re-estimate. A state's new mean is the average of all frames weighted
by that state's γ, its variance likewise, and the transition probabilities
come from the same soft counts. A frame the model is unsure about
contributes a little to several states rather than all of itself to a guess.

Then repeat. Each pass cannot lower the likelihood of the training data,
because re-estimation maximizes a lower bound on that likelihood which is
exactly tight at the current parameters: raising the bound either raises the
likelihood or leaves it where it is.

That is the EM guarantee, and it is why training can simply stop when the
likelihood stops moving, with no step size and no schedule to tune.

What EM does not promise is the *best* solution, only a stable one, which is
why the starting point in Flat initialization matters.

The demo below is a two-state, six-observation HMM, far smaller than any real
phone model but small enough to watch the mean move after a single update.

In [ ]:
obs = np.array([-1.1, -0.7, -0.2, 0.4, 0.9, 1.2])
mu = np.array([-0.8, 0.7])
var = np.ones(2) * 0.5
A = np.array([[0.75, 0.25], [0, 1.0]])


def fb():
    B = np.exp(-0.5 * (obs[:, None] - mu) ** 2 / var) / np.sqrt(2 * np.pi * var)
    a = np.zeros_like(B)
    a[0] = [1, 0] * B[0]
    sc = []
    for t in range(len(obs)):
        if t:
            a[t] = (a[t - 1] @ A) * B[t]
        sc.append(a[t].sum())
        a[t] /= max(sc[-1], 1e-300)
    bt = np.ones_like(B)
    for t in range(len(obs) - 2, -1, -1):
        bt[t] = A @ (B[t + 1] * bt[t + 1])
        bt[t] /= max(bt[t].sum(), 1e-300)
    g = a * bt
    g /= g.sum(1, keepdims=True)
    return g, np.log(sc).sum()


g, ll0 = fb()
old = mu.copy()
mu = (g * obs[:, None]).sum(0) / g.sum(0)
g, ll1 = fb()

fig, ax = plt.subplots(figsize=(7, 2.6))
img = ax.imshow(g.T, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_yticks([0, 1], ["state 1", "state 2"])
ax.set_xticks(range(len(obs)), [f"t{t}\n{o:+.1f}" for t, o in enumerate(obs)])
ax.set_xlabel("time step, with the observation value below it")
ax.set_title("Baum–Welch posterior occupancy γ — each column sums to 1")
fig.colorbar(img, ax=ax, label="P(state | all observations)")
plt.tight_layout()
plt.show()

print(f"mean before update: {old} -> after update: {mu}")
print(f"log-likelihood before: {ll0:.4f} -> after: {ll1:.4f}")
print(
    "Real training: mean = macc/dnom; var = vacc/dnom - mean^2; variance "
    "floor 1e-4; stop when per-frame ΔLL <= .001; CI/untied/tied schedules "
    "allow at most 10/6/10 iterations."
)

**How to read the figure.** Each column is one time step and each row is one
state, so a column holds that frame's posterior over states and sums to 1.
Darker means the model is more confident it was in that state at that time.

The transition matrix here is left-to-right with no way back
(`A = [[0.75, 0.25], [0, 1.0]]`), and the observations climb from -1.1 to
1.2 while the two state means start at -0.8 and 0.7. So expect the weight to
start on state 1, hand over once, and stay on state 2.

That handover is soft rather than a hard cut, and where it falls is exactly
what the mean update printed underneath responds to.

> **Checkpoint.** The `fb()` function above rescales `a[t]` and `bt[t]` at
> every time step (dividing by their sum) instead of working with raw
> forward/backward probabilities directly. Why?
>
> <details><summary>Answer</summary>
>
> Forward and backward probabilities are products of many terms less than
> 1, one per time step. Over a real utterance with hundreds of frames,
> that product underflows to exactly 0.0 in floating point long before the
> sequence ends.
>
> Rescaling at each step (and separately tracking the log of the scaling
> factors, `sc`, to recover the true log-likelihood) keeps every number in a
> representable range without changing the final posterior — this is the
> standard "scaled forward-backward" trick, and real implementations like
> `pstrain`'s do it for exactly this reason.
>
> </details>

> **Try it yourself.** What you will learn: that EM's monotonic improvement
> is a property of every pass, not just the first.
>
> What to change: at the end of the cell above, add a second mean update,
> `mu = (g * obs[:, None]).sum(0) / g.sum(0)`, then a third `fb()` call, and
> print its log-likelihood next to `ll0` and `ll1`.
>
> What to expect: a third value at least as large as the second. It may
> barely move, which is convergence rather than failure; real training stops
> on exactly that signal, when the per-frame improvement drops below
> 0.001.

## Gaussian splitting

**What.** A state's single Gaussian is replaced by two, then those two by
four, and so on until the state's mixture holds as many components as the
target density asks for.

**Why.** One Gaussian claims a state's frames form a single cloud with one
center. Real states are not like that.

The same phone state collects frames from different speakers, at different
speaking rates, in different neighboring contexts, and those form several
clusters rather than one. A single Gaussian stretched across all of them
fits none of them. A mixture can put a component on each.

**How.** Each Gaussian splits into two children whose means sit a small
fraction of a standard deviation to either side of the parent's mean, on
every dimension: `pstrain` uses ±0.2σ.

The children copy the parent's variance and take half its mixture weight
each, so together they keep the parent's total weight and start as a nearby,
symmetric pair around where it sat.

Baum-Welch then re-estimates, and the two children drift apart to cover
whatever the parent had been straddling. Repeat until the target density is
reached.

This is how the density ladder (`1g → 2g → 4g → ...`) grows without
restarting training from scratch at each rung.

In [ ]:
m = np.array([0.0, 0.0])
v = np.array([1.0, 0.25])
kids = np.stack([m - 0.2 * np.sqrt(v), m + 0.2 * np.sqrt(v)])

fig, ax = plt.subplots(figsize=(5.6, 5))
for kx, ky in kids:
    ax.annotate(
        "", xy=(kx, ky), xytext=(m[0], m[1]),
        arrowprops=dict(arrowstyle="-|>", linewidth=1.6, color="0.4",
                        shrinkA=11, shrinkB=11),
    )
ax.scatter(*m, s=220, color="#22456e", zorder=3, label="parent (weight 1.0)")
ax.scatter(kids[:, 0], kids[:, 1], s=150, color="#dd8452", zorder=3,
           label="children (weight 0.5 each)")
ax.set_xlim(-0.32, 0.32)
ax.set_ylim(-0.32, 0.32)
ax.set_aspect("equal")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.legend(loc="upper left", fontsize=9)
ax.set_title("Split: mean ± 0.2σ per dimension,\nweight halved, variance copied", fontsize=11)
plt.show()
print("children:", kids, "| weights: 0.5 / 0.5")

**How to read the figure.** Each arrow points from the parent Gaussian's
mean to one child's mean and reads as "goes to": the parent is replaced, not
joined.

The children are displaced by 0.2 standard deviations along every dimension,
which is why the vertical offset is smaller than the horizontal one here.
The variance in feature 2 is smaller.

The displacement is deliberately tiny. Its only job is to break the symmetry,
so that Baum-Welch has two distinguishable components to pull apart.

## Context and tying

**What.** A **triphone** is a phone modeled together with its immediate left
and right neighbors, so the `t` of "stop" and the `t` of "tree" become
different models. **Tying** is what makes that affordable: similar triphone
states are pooled and share one distribution, and a shared distribution of
that kind is called a **senone**.

**Why.** Neighboring phones change how a sound is produced and perceived.
The `t` in "stop" and in "tree" really are different stretches of acoustics,
and a model that treats them as one unit averages over a difference it could
have been using.

But modeling every triphone separately turns a few dozen units into
thousands, most seen only a handful of times, and a distribution estimated
from a handful of frames is worse than the average it replaced. Tying keeps
the distinctions the data can support and pools the rest.

**How.** `pstrain` trains the reachable untied triphones, then asks a series
of linguistically motivated yes/no *questions* about the context ("is the
left neighbor a liquid or a glide?").

It builds one decision tree per (phone, state), splitting each node on
whichever question buys the largest likelihood gain. It then globally prunes
the lowest-gain leaves down to the configured senone budget, and maps every
context to a surviving leaf, including contexts that never occurred in
training.

The demo below picks between two candidate questions using the same form of
likelihood-gain criterion `pstrain` uses internally. It splits the pooled
occupancy-weighted Gaussian into a "matches the question" group and a
"doesn't" group, and scores how much that split reduces overall variance,
weighted by how much data supports it.

The demo scores a single dimension; the real tree scores 39-dimensional
diagonal variances.

In [ ]:
contexts = np.array(["R", "L", "S", "K", "W", "N"])
occ = np.array([50, 35, 60, 40, 25, 45])
mm = np.array([1, 0.9, -0.7, -0.5, 1.2, -0.3])
vv = np.array([0.4, 0.5, 0.6, 0.5, 0.3, 0.7])
Q = {
    "liquids/glides": np.isin(contexts, ["R", "L", "W"]),
    "coronal/nasal": np.isin(contexts, ["S", "N"]),
}


def gain(mask):
    def cost(ix):
        w = occ[ix]
        m = (w * mm[ix]).sum() / w.sum()
        v = (w * (vv[ix] + (mm[ix] - m) ** 2)).sum() / w.sum()
        return 0.5 * w.sum() * np.log(v)

    return cost(np.ones(len(occ), bool)) - cost(mask) - cost(~mask)


scores = {k: gain(v) for k, v in Q.items()}
best = max(scores, key=scores.get)
winner = Q[best]
yes, no = occ[winner].sum(), occ[~winner].sum()

fig, (ax_gain, ax_split) = plt.subplots(1, 2, figsize=(11, 4))
ax_gain.bar(list(scores), list(scores.values()), color="#4c72b0")
ax_gain.set_ylabel("likelihood gain")
ax_gain.set_title("Candidate questions")
ax_split.pie(
    [yes, no],
    labels=[
        f"yes: {' '.join(contexts[winner])}\n{yes} frames",
        f"no: {' '.join(contexts[~winner])}\n{no} frames",
    ],
    autopct=lambda pct: f"{pct:.0f}%",
    startangle=90,
    counterclock=False,
    colors=["#4c72b0", "#c7c7c7"],
)
ax_split.set_title(f"Populations either side of: {best}")
plt.tight_layout()
plt.show()
print(scores)

**How to read the figure.** On the left, the bar for each candidate question
is how much likelihood its split would buy. The tree takes the taller bar
and never reconsiders, which is what "greedy" means here.

On the right are the groups that winning question actually creates, sized by
occupancy: the contexts answering yes on one side, the contexts answering no
on the other. Each group becomes a leaf, and every triphone state landing in
a leaf shares that leaf's senone.

A split that left one wedge nearly empty would buy a distinction almost no
data supports, which is what pruning removes later.

> **Try it yourself.** What you will learn: how much of a decision tree's
> quality is decided before the tree is ever built, by the question set it
> was handed.
>
> What to change: add a third entry to the `Q` dictionary in the cell above,
> for example `"stops/nasals": np.isin(contexts, ["K", "N"])`, and re-run.
> The available context symbols are printed in `contexts` at the top of that
> same cell.
>
> What to expect: a third bar, and possibly a new winner with a different
> yes/no split beside it. If a question you invented in a few seconds beats
> the phonetically motivated ones, that tells you how sensitive greedy tree
> building is to which questions it was offered in the first place, and why
> real systems use a carefully designed question set.

# Alignment and decoding

From here on the trained model is put to work. Forced alignment and
decoding are related, and easy to confuse, but they answer different
questions.

**Forced alignment.** *What*: given the audio and the words that were said,
find when each word and each phone occurred.

*Why*: the time stamps are the product, for phonetic research, for building
speech-synthesis training data, for subtitling, for anything that needs to
point at a moment in a recording.

*How*: the known words fix the phone sequence, so the only thing left to
search over is timing, and Viterbi finds the single best path through that
heavily constrained model.

**Decoding.** *What*: given the audio alone, find what was said. *Why*: this
is recognition. *How*: Viterbi beam search over a network built from the
lexicon and a language model, keeping only paths whose scores stay near the
best one so far, because the full network is far too large to explore
exhaustively.

## Viterbi, and how it differs from Baum-Welch

Both walk the same trellis of states against time, and both use the same
acoustic scores. They differ in the question they ask.

Baum-Welch, in training, asks how probable each state was at each frame and
keeps every answer. Viterbi asks which single sequence of states was most
probable and keeps only that one.

That difference is the right one in each place. Training wants soft counts,
because a genuinely ambiguous frame should contribute evidence to both
candidates rather than being forced onto one.

Alignment and decoding want an answer, and an answer is one path.

| | Baum-Welch (forward-backward) | Viterbi |
|---|---|---|
| question | how likely is each state at each frame? | which state sequence is most likely? |
| assignment | soft: a posterior over states, per frame | hard: one state per frame |
| combines paths by | summing over all of them | taking the maximum over them |
| output | occupancy counts for re-estimation | one best path |
| used for | training | forced alignment, decoding |

## Why HMM/GMM still matters

Newer acoustic models — DNN-HMM hybrids, end-to-end CTC and transducer
systems, self-supervised models such as wav2vec 2.0, and Whisper — achieve
lower word error rates on many benchmarks.

HMM/GMM nevertheless remains strong and widely used for **forced alignment**
and temporal localization: phone and word time stamps. It is the backbone of
major forced aligners including the Montreal Forced Aligner,
Prosodylab-Aligner, and Gentle.

Alignment constrains search to a known transcript, a small lattice of
possibilities rather than every possible sentence. That constraint lets a
simple, fast, CPU-only, deterministic, interpretable acoustic model localize
phones accurately, often well enough and far more cheaply than a large
end-to-end model.

The alignment overlay below is exactly this use case.

In [ ]:
from pstrain.api.alignment import align_corpus, save_ctm, save_textgrid

# A handful of utterances is enough to find a clean example for the overlay,
# and skipping the two the earlier sections listened to keeps this a new
# sentence to look at.
alignment_candidates = [
    (uid, text) for uid, text in train_transcripts.items()
    if uid not in (EXAMPLE_A, EXAMPLE_B)
]
alignment_subset = dict(alignment_candidates[:6])
job = align_corpus(
    alignment_subset,
    PROJECT / "audio",
    CI_MODEL_DIR,
    PROJECT / "shared/dictionary.dict",
    PROJECT / "shared/filler.dict",
    include_phones=True,
)
assert job.n_aligned > 0, job.errors

aligned_uid, alignment = next(iter(job.results.items()))
print(job.n_aligned, job.n_failed, aligned_uid, alignment.duration_time())
for segment in alignment.phones[:12]:
    print(
        segment.name,
        segment.start_time(alignment.frame_shift),
        segment.end_time(alignment.frame_shift),
    )

In [ ]:
aligned_waveform, aligned_sample_rate = librosa.load(
    PROJECT / "audio" / f"{aligned_uid}.wav", sr=None, mono=True
)
aligned_power = power_spectrogram(aligned_waveform, sr=aligned_sample_rate)

fig, ax = plt.subplots(figsize=(13, 4))
librosa.display.specshow(
    librosa.power_to_db(aligned_power, ref=np.max),
    sr=aligned_sample_rate, hop_length=HOP_LENGTH, x_axis="time", y_axis="hz",
    cmap="magma", ax=ax,
)
ax.set_ylim(0, 8000)
for segment in alignment.phones:
    start = segment.start_time(alignment.frame_shift)
    end = segment.end_time(alignment.frame_shift)
    color = "cyan" if segment.name == "SIL" else "white"
    ax.axvspan(start, end, color=color, alpha=0.07)
    ax.axvline(start, color="white", lw=0.4)
    ax.text(
        (start + end) / 2, 7500, segment.name,
        rotation=90, ha="center", va="top", fontsize=7, color="white",
    )
ax.set(title=f"CI-1G phone alignment — {aligned_uid}", xlabel="time (s)", ylabel="frequency (Hz)")
plt.show()

# Listen while reading the phone labels above: the boundaries the model
# chose are easiest to judge against what you actually hear.
display(Audio(aligned_waveform, rate=aligned_sample_rate))

**How to read the figure.** The background is the spectrogram of the
utterance the cell above aligned, with its id and words printed beside it.

Every white vertical line is a phone boundary the model chose, the label
above each band is the phone, and the tinted bands are the silences (`SIL`)
it found at the edges and in any pause.

Listen while you read the labels. Boundaries at a stop closure or at the
start of a fricative should land where you hear the change.

Boundaries between a vowel and a following nasal or glide are the hard ones,
and the model placing one a frame or two early or late is normal rather than
a fault.

Nothing here was hand-labeled. The transcript said which phones came in what
order, and the model decided where.

Alignments are written out in two standard formats. A **CTM** file is a flat
list of timed labels, one per line, which is what most alignment tooling
reads. A **TextGrid** is Praat's tiered format, which opens directly in
Praat next to the waveform.

In [ ]:
CTM = WORK / f"{aligned_uid}.phones.ctm"
TG = WORK / f"{aligned_uid}.TextGrid"
save_ctm(alignment, CTM, level="phones")
save_textgrid(alignment, TG)
print("\n".join(CTM.read_text().splitlines()[:8]))
print("\nTextGrid:\n" + "\n".join(TG.read_text().splitlines()[:8]))

## Decoder ingredients

**What.** A decoder needs three things: an acoustic model, a lexicon, and a
language model. We have the first two already; the third is built below.

**Why.** Acoustics alone cannot choose between "recognize speech" and "wreck
a nice beach", which sound nearly identical. A language model, here an
n-gram counting how often each short word sequence occurs in the training
text, supplies the prior that settles it. The lexicon supplies the
vocabulary: a decoder can only ever output words it has pronunciations for.

**How.** PocketSphinx loads means, variances, mixture weights, transition
matrices, `mdef`, `feat.params`, and optionally a compressed `sendump`.

A forward-tree, then forward-flat, then best-path beam search combines the
GMM/HMM acoustics with the dictionary pronunciations and an **ARPA** n-gram
language model, the plain-text format that lists each n-gram with its log
probability and its back-off weight.

Defaults include beam `1e-80`, word beam `1e-40`, LM weight 10, and
insertion penalty 0.2.

Because we have trained on only `N_UTTS` utterances, we build a matching
n-gram language model directly from the training transcripts, so decoding
always has an explicit, matched LM rather than an open-vocabulary one.

Accuracy is reported as **word error rate** (WER): substitutions plus
insertions plus deletions, divided by the number of words in the reference.
It can exceed 1.0, because a decoder can insert more words than were
actually said.

In [ ]:
from pstrain.api import build_lm

LM = build_lm(list(train_transcripts.values()), WORK / "train.arpa", 3)
assert Path(LM).is_file()
print(LM, Path(LM).stat().st_size, "bytes")

In [ ]:
from pstrain.api.testing import test_model

decode = test_model(
    MODEL_DIR,
    PROJECT / "audio",
    test_transcripts,
    PROJECT / "shared/dictionary.dict",
    PROJECT / "shared/filler.dict",
    lm=LM,
    verbose=True,
    jobs=JOBS,
)
assert decode.n_decoded == len(test_transcripts)
DECODE_WER = decode.wer
print(f"decoded={decode.n_decoded}/{decode.n_utterances}  WER={DECODE_WER:.3f}")

# Held-out sentences: none of these was seen during training, and none is a
# sentence any earlier figure in this notebook used.
for u, z in list(decode.per_utterance.items())[:3]:
    print(u, "\n  REF:", z["reference"], "\n  HYP:", z["hypothesis"])


# Cross-check pstrain's own reported WER against an independent, widely-used
# implementation (jiwer), computed over the same reference/hypothesis pairs.
references = [z["reference"] for z in decode.per_utterance.values()]
hypotheses = [z["hypothesis"] for z in decode.per_utterance.values()]
JIWER_WER = jiwer.wer(references, hypotheses)
JIWER_CER = jiwer.cer(references, hypotheses)
print(f"\njiwer cross-check: WER={JIWER_WER:.3f}  CER={JIWER_CER:.3f}")
print(
    "pstrain's WER and jiwer's WER should match (both are standard "
    "substitutions+insertions+deletions / reference-word-count); a "
    "mismatch would point to a tokenization difference, not a real "
    "disagreement about accuracy."
)

print(
    "\nHigh WER is expected from a 1-Gaussian model trained on a small "
    "corpus with a tiny, matched LM; this is a green learning spine, not "
    "a benchmark result."
)

> **Checkpoint.** Word error rate (WER) here can look alarmingly high —
> often well above what you'd see from a commercial system. Given
> everything you now know about this pipeline (corpus size, Gaussian
> count, LM size), is that surprising?
>
> <details><summary>Answer</summary>
>
> No — every lever that improves accuracy was set for speed, not accuracy,
> in Configuration:
>
> - `cd-1g` has only one Gaussian per senone (real deployments commonly use
>   8–32);
> - `N_UTTS=150` is a tiny fraction of the corpus's ~1,132 utterances;
> - the language model is trained on the same small set of sentences.
>
> Each of these is independently a large lever on WER. The point of this
> configuration is a complete, fast, correct pipeline you can inspect
> end-to-end, not a competitive accuracy number.
>
> </details>

> **Try it yourself.** What you will learn: how far the two cheapest levers,
> data and density, move a real error rate.
>
> What to change: `N_UTTS` and `TARGET` in the configuration cell. Try
> `N_UTTS = 600` with `TARGET = "cd-8g"`, or `N_UTTS = None` for the whole
> corpus, then re-run the notebook from Configuration onward.
>
> What to expect: a noticeably longer training cell, since training scales
> with both data size and Gaussian count, and a meaningfully lower WER at
> the end. Watch the packaged smoke test at the very bottom still agree with
> the unpackaged number: that agreement is the check that nothing about the
> change broke the deployable artifact.

# Model capacity and configuration

`pstrain` ships several built-in configuration profiles: `default`,
`wideband`, `telephone`, `wideband_large`, `sphinxtrain`.

Settings are resolved with precedence built-in < user < project <
experiment < CLI — each level can override the one before it.

Inspect any profile with `resolve_config(...).as_dict()` as we've been
doing, or from a terminal with
`pstrain config show/get/explain/list/schema`.

In [ ]:
large = resolve_config(PROJECT, profile_name="wideband_large").as_dict()
print("default senones:", cfg["training"]["n_senones"],
      "| wideband_large senones:", large["training"]["n_senones"])

preview = PipelineContext.from_config(
    PROJECT,
    experiment="senone-preview",
    config_name="default",
    cli_overrides={"training": {"n_senones": 100}},
)
assert build_pipeline(preview).run("cd-1g", dry_run=True, jobs=JOBS) == 0
print("Separate 100-senone experiment previewed; no retraining performed.")

In [ ]:
telephone_features = resolve_config(PROJECT, profile_name="telephone").as_dict()["features"]
default_features = cfg["features"]

print(f"{'setting':12s} {'default':>10s} {'telephone':>10s}")
for field in ("samprate", "nfilt", "nfft", "lowerf", "upperf"):
    print(
        f"{field:12s} {str(default_features[field]):>10s} "
        f"{str(telephone_features[field]):>10s}"
    )
print("An 8 kHz channel trades bandwidth/resolution; changing features requires retraining.")

## Capacity guide

Context-independent (CI) models are data-efficient; context-dependent (CD)
models capture neighboring-phone effects at a higher data cost.

Untied CD is sparse (many contexts seen only rarely); tree-tied senones
share evidence across similar contexts to make that data go further.

More Gaussians per state model more acoustic sub-modes but cost compute and
risk overfitting with too little data. More senones preserve context detail
but likewise need more data to estimate reliably.

CI/untied/tied training schedules allow up to 10/6/10 Baum-Welch iterations
each and stop early once per-frame likelihood improvement drops to 0.001 or
below.

## The recognition target

The pipeline is unit-agnostic: you supply the phoneset, lexicon,
transcripts, and audio, and the model learns whatever units the lexicon
maps words onto. That gives wide latitude:

- **Phonemes** — the units used throughout this notebook.
- **Stress or no stress** — CMUdict marks vowel stress with digits such as
  `AH0`, `AH1`, `AH2`. Keeping them makes each stressed vowel a distinct
  unit: more context detail, a larger inventory, and a greater data
  requirement. This notebook's dictionary strips stress for a smaller
  inventory, as Pronunciations and Dictionaries showed.
- **Visemes** — mouth-shape classes form a reduced-cardinality unit set for
  lip-reading or talking-head animation, demonstrated below.
- **Another language** — swap the phoneset, lexicon, and training data,
  then choose a matching configuration profile (e.g. the appropriate
  sample rate). The training machinery is unchanged.

To build a viseme model for real, you would map phones to visemes, rewrite
every dictionary pronunciation as a viseme sequence, derive the phoneset
from that rewritten lexicon, and run the **same pipeline**.

Fewer units means more training examples per unit, so decision-tree training
is easier and faster than for the phoneme model. Recall from Context and
tying that rare-unit tree failures happen when a unit has too little data.

In [ ]:
# One simplified mapping; real inventories vary (for example, Disney 12,
# Jeffers-Barley, and MPEG-4).
VISEME = {
    "SIL": "sil",
    "P": "PBM", "B": "PBM", "M": "PBM",
    "F": "FV", "V": "FV",
    "TH": "TH", "DH": "TH",
    "T": "TDNL", "D": "TDNL", "N": "TDNL", "L": "TDNL",
    "S": "SZ", "Z": "SZ",
    "SH": "SH", "ZH": "SH", "CH": "SH", "JH": "SH",
    "K": "KG", "G": "KG", "NG": "KG", "HH": "KG",
    "R": "R", "W": "W", "Y": "Y",
    "IY": "IY", "IH": "IY",
    "EH": "EH", "EY": "EH", "AE": "EH",
    "AA": "AA", "AH": "AA", "AO": "AA",
    "UW": "UW", "UH": "UW", "OW": "UW",
    "AW": "AY", "AY": "AY", "OY": "AY",
    "ER": "ER",
}


def collapse_repeats(units):
    '''Collapse adjacent identical units, as many real viseme lexicons do.'''
    collapsed = []
    for unit in units:
        if not collapsed or unit != collapsed[-1]:
            collapsed.append(unit)
    return collapsed


# Transform every pronunciation, including word(2), word(3), and later variants.
viseme_lexicon = {
    word: collapse_repeats([VISEME[phone] for phone in pronunciation])
    for word, pronunciation in L.items()
}
viseme_phoneset = sorted(
    {viseme for pronunciation in viseme_lexicon.values() for viseme in pronunciation}
    | {"sil"}
)

print("Phoneme inventory:", len(phones))
print("Viseme inventory:", len(viseme_phoneset))
print("Reduction:", len(phones) - len(viseme_phoneset), "units")
print("Viseme phoneset:", " ".join(viseme_phoneset))

for word in ("the", "author", "of"):
    ph = L[word]
    vi = viseme_lexicon[word]
    print(f"{word:>6}: {' '.join(ph):<20} -> {' '.join(vi)}")

print(
    "To train this for real: write the viseme lexicon (and optionally the "
    "viseme phoneset) to files, point setup_project at them, and run the "
    "same pipeline."
)

> **Try it yourself.** What you will learn: why lip-reading is ambiguous in
> a way that listening is not.
>
> What to change: the word list in the loop at the end of the cell above.
> Any key in `L` works; `sorted(L)[:20]` prints some to choose from.
>
> What to expect: pick a minimal pair, `"bat"` against `"pat"`, and print
> both. The phone sequences differ in their first phone; the viseme
> sequences are identical, because `B` and `P` are made with the same mouth
> shape and differ only in voicing, which the lips do not show. That
> collapse is exactly what a lip-reader has to resolve from context.

## The density ladder

Each split doubles the number of Gaussians per state, Baum-Welch then
retrains the enlarged model, and the pair repeats until the target density
is reached: `1g` to `2g` to `4g`, on up to `32g`. Retraining is the half
that matters, because a split on its own changes nothing: it is what the
children do afterward that buys the accuracy.

The training likelihood is what that looks like from the inside, so the
figure below plots it per pass, from the telemetry this notebook's own run
wrote.

In [ ]:
import json

curves = []
without_telemetry = []
without_passes = []
for stage_dir in sorted(p for p in (PROJECT / "shared/models").iterdir() if p.is_dir()):
    path = stage_dir / "default/bw_telemetry.json"
    if not path.is_file():
        without_telemetry.append(stage_dir.name)
        continue
    passes = json.loads(path.read_text()).get("passes") or []
    if not passes:
        without_passes.append(stage_dir.name)
        continue
    # Modification order is the order the stages ran, which is the order the
    # ladder climbs; sorting by name would scramble it.
    curves.append((path.stat().st_mtime, stage_dir.name, passes))
curves.sort(key=lambda item: item[0])

if curves:
    fig, ax = plt.subplots(figsize=(8, 4))
    for _, stage, passes in curves:
        ax.plot(
            [p["pass"] for p in passes],
            [p["per_frame_log_likelihood"] for p in passes],
            marker="o",
            label=stage,
        )
    ax.set(
        xlabel="Baum-Welch pass",
        ylabel="per-frame log-likelihood",
        title="What retraining buys, one stage at a time",
    )
    ax.legend()
    plt.show()

for _, stage, passes in curves:
    print(
        f"{stage:>10}: {len(passes)} pass(es), "
        f"{passes[0]['per_frame_log_likelihood']:.1f} -> "
        f"{passes[-1]['per_frame_log_likelihood']:.1f}"
    )
# Say what is not plotted rather than dropping it quietly: a stage that
# writes no telemetry ran no Baum-Welch passes, which is a fact about the
# stage, not a gap in the figure.
if without_telemetry:
    print("no Baum-Welch telemetry:", ", ".join(without_telemetry))
if without_passes:
    print("telemetry present but no passes recorded:", ", ".join(without_passes))
if not curves:
    print("Nothing to plot yet; run the training cell first.")

**How to read the figure.** One line per training stage, each point a
Baum-Welch pass, and higher is a better fit to the training audio.

Each line should be nondecreasing, up to numerical effects, because no pass
can make the fit worse. That is the EM guarantee in the large. The flattening
out is what the convergence threshold watches for, and a stage that exhausts
its iteration budget before flattening simply stops where it got to.

The lines do not join up, because each stage is a different model scored
against the same audio.

The untied stage climbs furthest, since a private distribution per triphone
fits the training data better than anything else here can. The tied stage
after it gives much of that back deliberately: tying trades training-set fit
for distributions estimated from enough data to survive contact with speech
the model has not heard.

# Packaging and deployment

Deployment needs the acoustic parameters and `feat.params`, the main and
filler dictionaries, and an explicit ARPA language model.

Always test the *exact* package you plan to distribute — packaging can
subtly change paths or file layout, and a smoke test catches that
immediately rather than in production.

In [ ]:
from pstrain.api import package_model

PROOT = WORK / "package"
packaged = package_model(
    MODEL_DIR,
    PROOT,
    model_name="arctic-tutorial",
    dictionary_path=PROJECT / "shared/dictionary.dict",
    filler_dict_path=PROJECT / "shared/filler.dict",
    include_dict=True,
)
PACKAGE = PROOT / "arctic-tutorial"
print(packaged)
for q in sorted(PACKAGE.rglob("*")):
    if q.is_file():
        print(" ", q.relative_to(PACKAGE), q.stat().st_size)

In [ ]:
packaged_decode = test_model(
    PACKAGE / "acoustic",
    PROJECT / "audio",
    test_transcripts,
    PACKAGE / "dict/cmudict.dict",
    PACKAGE / "dict/filler.dict",
    lm=LM,
    verbose=True,
    jobs=JOBS,
)
assert packaged_decode.n_decoded == len(test_transcripts)
assert abs(packaged_decode.wer - DECODE_WER) < 1e-12
print(f"PACKAGED_SMOKE_WER={packaged_decode.wer:.3f}")

u, z = next(iter(packaged_decode.per_utterance.items()))
print(u, "\n  REF:", z["reference"], "\n  HYP:", z["hypothesis"])
print("OK: packaged model decodes successfully and matches the unpackaged WER")

# Recap

Spectrograms expose time-frequency energy; MFCCs compress it into a compact,
near-decorrelated feature vector, and the cepstral chain is what separates
the vocal-tract envelope from the pitch that carried it.

A phoneme is a contrastive category in a language and a phone is a concrete
realization of one; the ARPAbet units we modeled are called phones by
convention. HMM states are temporal phases within a phone model, senones are
shared tied-state distributions, and Gaussians are mixture components within
a senone.

Context-independent models ignore neighboring phones; context-dependent
models use decision-tree-tied triphones to share evidence across similar
contexts.

Baum-Welch (an EM instance) trains the model from fractional occupancy;
Viterbi commits to one best path, and both force-aligns known transcripts
and decodes unknown audio.

The target name selects the model family and Gaussian density; configuration
profiles select features, training schedules, data splits, and senone
budgets. Every input this notebook used, it either downloaded or generated
itself.

HMM/GMM remains especially valuable for fast, interpretable forced
alignment when the transcript is known and accurate time stamps matter,
even as end-to-end neural models lead on open-vocabulary recognition
accuracy.

**Next, on your own:**

- Raise `N_UTTS` toward the corpus's full ~1,132 utterances and try
  `cd-8g`, the toolkit default, to see how much WER improves.
- Train a larger n-gram language model from more text and see how much of
  the WER gap that alone closes versus improving the acoustic model.
- Adjust `n_senones` only once you've raised the data size to support it,
  as Model capacity and configuration showed; watch for decision-tree
  warnings on rare phones.
- Rerun the packaged smoke test after any change: it is your fastest signal
  that a change didn't silently break the deployable artifact.

## Glossary

Every term below is introduced where it is first needed; this is a recap,
not a first encounter.

| term | meaning |
|---|---|
| ARPA LM | an n-gram language model in the standard ARPA text format, listing each n-gram with its log probability and back-off weight |
| Baum-Welch | the EM algorithm instance used to train HMM/GMM parameters from audio with no state labels on it |
| cepstrum | the inverse Fourier transform of a log-magnitude spectrum; the coinage, and why the operation approximately separates vocal tract from pitch, are in From spectra to MFCCs |
| CMN | cepstral mean normalization: subtracting the per-utterance (or per-batch) mean cepstrum to reduce channel and speaker effects |
| CTM / TextGrid | standard file formats for reporting timed alignments (a flat list of labels / Praat-style tiers) |
| decision-tree state tying | the algorithm that decides which triphone states share a senone, via greedy phonetic-question splits |
| decoding | using Viterbi beam search with an unknown transcript, guided by a lexicon and a language model |
| flat initialization | the starting point for training: one global Gaussian tiled into every state, before any specialization |
| forced alignment | using Viterbi with a known transcript to find phone and word time stamps |
| forward-backward | the recursion that computes each state's fractional (soft) occupancy at every time step |
| frame | a short (~25 ms), overlapping (10 ms hop) slice of audio treated as approximately stationary |
| Gaussian splitting | growing model capacity by splitting one Gaussian into two placed either side of its mean |
| GMM | Gaussian mixture model, which models the feature distribution within one HMM state |
| HMM | hidden Markov model, which models how acoustic state changes over time |
| lexicon / dictionary | the mapping from words to one or more phone-sequence pronunciations |
| mel filterbank | a bank of frequency filters spaced to match human pitch perception (dense at low frequencies, sparse at high) |
| MFCC | mel-frequency cepstral coefficient: the compact, near-decorrelated feature vector computed from log-mel energies via a DCT |
| monophone (CI) | a phone modeled without regard to its neighbors, that is, context-independent |
| multipron | training with more than one pronunciation path per word active simultaneously in the HMM graph |
| phone | a concrete realization of a phoneme, an actual stretch of speech sound; the ARPAbet symbols this notebook models are conventionally called phones |
| phoneme | the abstract category that distinguishes meaning in a language: swap one and you have said a different word |
| senone | a shared, tied-state distribution: the result of clustering similar triphone states together |
| STFT | short-time Fourier transform: the frame-by-frame spectrum that makes a spectrogram |
| triphone (CD) | a phone modeled in the context of its immediate left and right neighbors, that is, context-dependent |
| Viterbi | the dynamic-programming algorithm that finds the single most likely state sequence, used for alignment and decoding |
| WER | word error rate: substitutions plus insertions plus deletions, divided by the reference word count |